<br>

<center>

| <!-- -->   | <!-- -->   | <!-- -->  |
|:------------- |:---------------:| -------------:|
| <img src="https://www.iimas.unam.mx/wp-content/uploads/2023/11/Logo-pagina-ok-800x248.png" alt="JuveYell" width="350px"> | **Programa - Búsqueda de palabras clave en OA de COVID** <br> Fecha: 28/02/2026. <br> Proyecto: PAPIIT IN302623 - Indicadores de ciencia y tecnología en el contexto de la Ciencia Abierta <br> Bases de datos sobre COVID. <br> <br> Dr. Eduardo Robles Belmont <br> M. en C. José Gerardo López Bonifacio. <br> Script R para OA de COVID MX completo | <img src="https://media.licdn.com/dms/image/C560BAQExBAK79vk3NA/company-logo_200_200/0/1654532058315?e=2147483647&v=beta&t=czCmWVT97xQXzW0hdloOgat3Q7G-9itqMuYQbu3KzbI" alt="" width="0px"> |

</center>

<br>

# Explicación detallada del script de consulta a OpenAlex (COVID-19 México)

Este script en **R** automatiza la consulta al endpoint `/works` de OpenAlex para recuperar publicaciones relacionadas con COVID-19 y coronavirus, aplicando filtros que garantizan que los trabajos estén vinculados con México.

El flujo general del programa es:

Consulta paginada → Extracción estructurada → Limpieza controlada → Filtro fuerte por institución mexicana → Deduplicación global → Exportación a CSV → Compresión en ZIP → Generación de log.

Está diseñado para ejecutarse en Google Colab (directorio `/content/`) y contiene mecanismos explícitos para evitar errores por consumo excesivo de memoria.

---

# 1) Instalación y carga de dependencias

El script declara un vector de paquetes necesarios y:

1. Verifica si cada uno está instalado.
2. Instala los faltantes desde CRAN.
3. Los carga silenciosamente.

Paquetes utilizados:

- httr: solicitudes HTTP al API.
- jsonlite: serialización y manejo de estructuras JSON.
- purrr: iteración funcional sobre listas.
- dplyr: transformación y filtrado de datos.
- tibble: creación de dataframes modernos.
- stringr y stringi: limpieza y normalización de texto.
- readr: lectura y escritura eficiente de CSV.
- zip: compresión de archivos.

Se deja además una opción comentada para usar `arrow` (formato Parquet) en caso de requerirse mayor eficiencia de almacenamiento.

---

# 2) Definición del universo de búsqueda

## 2.1 Lista de términos

El vector `terminos` incluye variantes léxicas relevantes:

- COVID-19  
- COVID19  
- Coronavirus  
- Corona virus  
- 2019-nCoV  
- SARS-CoV2  
- SARS-CoV-2  
- SARS-CoV  
- MERS-CoV  
- Severe Acute Respiratory Syndrome  
- Middle East Respiratory Syndrome  

El objetivo es identificar las referencias bibliográficas de productos de investigaciones académicas sobre el virus Covid-19 en títulos y resúmenes.

## 2.2 Carpeta de salida

Se crea la carpeta:

`/content/por_termino_covid_mx`

Aquí se almacenarán:

- CSV curados por término.
- CSV FULLFLAT (aplanado completo).
- Archivos intermedios.

---

# 3) Control de resultados

Se inicializan dos vectores:

- `terminos_con_resultados`
- `terminos_sin_resultados`

Estos permiten generar un resumen final estructurado y un archivo de log.

---

# 4) Funciones auxiliares (robustez estructural)

Esta sección resuelve dos problemas frecuentes al trabajar con APIs:

1. Campos que pueden venir como NULL.
2. Estructuras JSON profundamente anidadas.

## 4.1 Operador de valor por defecto

Se define un operador que devuelve un valor alternativo cuando el original es NULL.  
Esto evita errores al mapear listas incompletas.

## 4.2 Colapsadores seguros

Se implementan funciones que:

- Aplanan listas a strings separados por punto y coma.
- Serializan estructuras complejas a JSON cuando no pueden representarse como valores escalares.

Esto previene errores de escritura en CSV y evita resultados ambiguos.

## 4.3 Reconstrucción del abstract

OpenAlex entrega el abstract como índice invertido (palabra → posiciones).

La función `decode_abstract`:

1. Calcula la longitud máxima del texto.
2. Inserta cada palabra en su posición correspondiente.
3. Reconstruye el texto final.

El resultado es un abstract legible y continuo.

## 4.4 Conversión segura a formato CSV

Se define una función que convierte cualquier objeto en un string compatible con CSV:

- Listas → JSON
- Vectores múltiples → colapsados con "; "
- Vacíos → NA

## 4.5 Aplanado completo del JSON

Las funciones de flatten:

- Recorren recursivamente la estructura del work.
- Generan claves jerárquicas tipo campo.subcampo.valor.
- Devuelven una fila completamente plana.

Esto se usa en el modo FULLFLAT.

## 4.6 Pickers tipados

Se definen funciones que fuerzan:

- Un solo carácter (pick_chr1)
- Un solo entero (pick_int1)
- Un solo double (pick_dbl1)
- Un booleano (pick_lgl1)

Su propósito es evitar que columnas cambien de tipo o generen errores de longitud mayor a 1.

---

# 5) Filtro fuerte por institución mexicana

Se implementa una función que verifica:

- Si al menos un autor
- Tiene al menos una institución
- Con country_code == "MX"

Este filtro es más estricto que el filtro inicial del API y asegura consistencia metodológica.

---

# 6) Escritura FULLFLAT por bloques (control de RAM)

Para evitar consumir toda la memoria:

1. Se divide la lista de resultados en bloques (por defecto 200).
2. Cada bloque se aplana y se escribe inmediatamente en disco.
3. Se limpia memoria con `gc()`.

Esto impide la creación de un dataframe gigante en RAM.

---

# 7) Limpieza controlada de texto

Se aplica limpieza solo a columnas humanas:

- title
- abstract
- authors
- institutions
- countries
- host_venue
- display_name

Transformaciones:

- Conversión a minúsculas.
- Eliminación de acentos.
- Eliminación de HTML.
- Normalización de espacios.
- Eliminación de caracteres no imprimibles.

No se limpian columnas JSON para evitar corrupción estructural.

---

# 8) Deduplicación global entre términos

Se utiliza un environment como tabla hash:

- Si un ID ya fue visto en otro término → se descarta.
- Si es nuevo → se conserva.

Además, se aplica `distinct(id)` como protección adicional.

---

# 9) Consulta paginada con cursor

Para cada término:

1. Se construye la URL con:
   - title_and_abstract.search
   - filtro authorships.countries:MX
   - per-page=200
   - cursor=*

2. Se ejecuta un ciclo repeat que:
   - Descarga resultados.
   - Acumula páginas.
   - Actualiza el cursor.
   - Se detiene cuando no hay más resultados.

Incluye pausa de 1 segundo por página para evitar sobrecargar el servidor.

---

# 10) Construcción del dataframe curado

Se construye un tibble que incluye:

- Identificadores (id, doi, ids_openalex, etc.)
- Metadatos de publicación
- Abstract reconstruido
- Métricas de citación
- Información de open access
- Locations y venues
- Autorías e instituciones
- Tópicos, conceptos, keywords, SDG
- Referencias y trabajos relacionados
- Información de APC y financiamiento
- Conteos por año
- Flags de contenido
- JSON completo del work
- Metadatos del endpoint (meta_count, meta_page, etc.)

Este dataframe es legible y analíticamente utilizable.

---

# 11) Exportación por término

Por cada término se generan:

1. Archivo FULLFLAT completo:
   `/content/por_termino_covid_mx/<termino>_OpenAlex_MX_FULLFLAT.csv`

2. Archivo curado:
   `/content/por_termino_covid_mx/<termino>_OpenAlex_MX.csv`

Si no hay registros válidos tras filtros, el término se registra como sin resultados.

---

# 12) Compresión y log final

## 12.1 Archivo ZIP

Se listan todos los CSV generados y se crea:

`/content/resultados_COVID_MX.zip`

## 12.2 Archivo de log

Se genera:

`/content/log_busqueda_COVID_MX.csv`

Contiene:

- término
- ruta del archivo
- número de resultados
- indicador booleano de existencia

---

# Resultados esperados

Al finalizar la ejecución, el directorio `/content/` contendrá:

1. Carpeta `/content/por_termino_covid_mx/`
2. Archivo `/content/resultados_COVID_MX.zip`
3. Archivo `/content/log_busqueda_COVID_MX.csv`

---

# Garantías metodológicas del script

El script asegura:

- Consulta exhaustiva por paginación.
- Filtro fuerte por institución mexicana.
- Deduplicación global.
- Conservación del JSON original.
- Control explícito de memoria.
- Registro reproducible mediante log.

No asegura:

- Exhaustividad semántica absoluta del término (depende de título y abstract).
- Equivalencia exacta entre filtro API y filtro institucional interno.
- Que el FULLFLAT sea legible; su objetivo es completitud estructural.


In [ ]:
# ::::::::::::::::::::::::::::::::::::::::: CARGA DE PAQUETES :::::::::::::::::::::::::::::::::::::::::
# Aquí se listan las bibliotecas (paquetes) que el script necesita para funcionar.
# Si alguno no está instalado, el código lo instala automáticamente y luego los carga todos en memoria.
dependencias <- c("httr", "jsonlite", "purrr", "dplyr", "tibble", "stringr", "readr", "zip", 'stringi')
invisible(lapply(dependencias, function(p) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
  suppressPackageStartupMessages(library(p, character.only = TRUE))
}))
invisible(lapply(dependencias, library, character.only = TRUE))

# =========================
# RECOMENDACIÓN: opcional Parquet (NO rompe si no está)
# (si lo quieres, descomenta "arrow" en dependencias y estas líneas)
# =========================
# if (!requireNamespace("arrow", quietly = TRUE)) install.packages("arrow", repos = "https://cloud.r-project.org")
# suppressPackageStartupMessages(library(arrow))

# ::::::::::::::::::::::::::::::::::::::::: LISTA DE TÉRMINOS :::::::::::::::::::::::::::::::::::::::::::::
# Palabras clave que se van a buscar en OpenAlex relacionadas con COVID-19 y coronavirus.
# Me tomé el atrevimiento de agregar a esa ecuación de búsqueda "SARS-CoV2" y "SARS-CoV-2".
# Pues he visto artículos y documentos en varios lugares con esos términos también.
terminos <- c(
  "COVID-19", "COVID19", "Coronavirus", "Corona virus", "2019-nCoV",
  "SARS-CoV2", "SARS-CoV-2", "SARS-CoV", "MERS-CoV",
  "Severe Acute Respiratory Syndrome", "Middle East Respiratory Syndrome"
)

# ::::::::::::::::::::::::::::::::::::::::: CREAR CARPETA DE SALIDA :::::::::::::::::::::::::::::::::::::::::
# Se crea una carpeta donde se guardarán los resultados para cada término.
dir.create("/content/por_termino_covid_mx", showWarnings = FALSE)

# ::::::::::::::::::::::::::::::::::::::::: CONSULTAS :::::::::::::::::::::::::::::::::::::::::::::::::::::::
# Se preparan listas para guardar qué términos dieron resultados y cuáles no.
terminos_con_resultados <- c()
terminos_sin_resultados <- c()

# =========================
# Helpers globales (FUERA del for)
# =========================

`%||%` <- function(x, y) if (is.null(x)) y else x

safe_collapse <- function(x) {
  if (is.null(x)) return(NA_character_)
  x <- unlist(x, recursive = TRUE, use.names = FALSE)
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA_character_)
  paste(unique(as.character(x)), collapse = "; ")
}

safe_collapse_kv <- function(x) {
  if (is.null(x)) return(NA_character_)
  # counts_by_year suele ser lista de listas; si llega como named, lo serializamos
  jsonlite::toJSON(x, auto_unbox = TRUE, null = "null")
}

# ::::::::::::::::::::::::::::::::::::::::: FUNCIÓN PARA DECODIFICAR ABSTRACT :::::::::::::::::::::::::::::::
# Esta función reconstruye los resúmenes (abstracts) que vienen codificados palabra por palabra en OpenAlex.
# Básicamente convierte una estructura compleja de datos en texto legible.
decode_abstract <- function(inv_index) {
  if (is.null(inv_index)) return(NA_character_)
  inv_values <- unlist(inv_index)
  if (length(inv_values) == 0) return(NA_character_)
  words <- character(max(inv_values) + 1)
  for (word in names(inv_index)) {
    for (i in inv_index[[word]]) words[i + 1] <- word
  }
  paste(words, collapse = " ")
}

csv_safe_value <- function(v) {
  if (is.null(v)) return(NA_character_)

  if (inherits(v, "data.frame")) {
    return(as.character(jsonlite::toJSON(v, auto_unbox = TRUE, null = "null")))
  }

  if (is.list(v)) {
    return(as.character(jsonlite::toJSON(v, auto_unbox = TRUE, null = "null")))
  }

  if (is.atomic(v)) {
    if (length(v) == 0) return(NA_character_)
    if (length(v) == 1) return(as.character(v))
    return(paste(as.character(v), collapse = "; "))
  }

  return(as.character(v))
}

flatten_to_named_list <- function(x, prefix = "") {
  out <- list()

  if (!is.list(x) || is.data.frame(x)) {
    nm <- ifelse(prefix == "", "value", prefix)
    out[[nm]] <- x
    return(out)
  }

  nms <- names(x)
  if (is.null(nms)) {
    nm <- ifelse(prefix == "", "value", prefix)
    out[[nm]] <- x
    return(out)
  }

  for (k in nms) {
    v <- x[[k]]
    key <- if (prefix == "") k else paste0(prefix, ".", k)

    if (inherits(v, "data.frame")) {
      out[[key]] <- v
    } else if (is.list(v) && is.null(names(v))) {
      out[[key]] <- v
    } else if (is.list(v)) {
      out <- c(out, flatten_to_named_list(v, key))
    } else {
      out[[key]] <- v
    }
  }

  out
}

work_to_flat_row <- function(work) {
  flat <- flatten_to_named_list(work)

  # convierte cada elemento a string SIEMPRE
  flat_chr <- purrr::map(flat, csv_safe_value)
  flat_chr <- purrr::map(flat_chr, as.character)

  # tibble 1 fila, todo character
  tibble::as_tibble(flat_chr)
}

# =========================
# Helpers EXTRA (recomendados)
# =========================

# Devuelve SIEMPRE 1 string (o NA). Si hay >1, colapsa con "; ".
pick_chr1 <- function(x) {
  if (is.null(x)) return(NA_character_)
  if (is.list(x) && !is.data.frame(x)) {
    # si es lista con nombres o lista de cosas -> JSON para no perder estructura
    if (!is.null(names(x))) return(jsonlite::toJSON(x, auto_unbox = TRUE, null = "null"))
    x <- unlist(x, recursive = TRUE, use.names = FALSE)
  }
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA_character_)
  if (length(x) == 1) return(as.character(x))
  paste(as.character(x), collapse = "; ")
}

pick_int1 <- function(x) {
  if (is.null(x) || length(x) == 0 || all(is.na(x))) return(NA_integer_)
  suppressWarnings(as.integer(x[[1]]))
}

pick_dbl1 <- function(x) {
  if (is.null(x) || length(x) == 0 || all(is.na(x))) return(NA_real_)
  suppressWarnings(as.numeric(x[[1]]))
}

pick_lgl1 <- function(x, default = FALSE) {
  if (is.null(x) || length(x) == 0 || all(is.na(x))) return(default)
  as.logical(x[[1]])
}

# Serializa a JSON (1 string) para objetos/listas nombradas
safe_json <- function(x) {
  if (is.null(x)) return(NA_character_)
  as.character(jsonlite::toJSON(x, auto_unbox = TRUE, null = "null"))
}

# =========================
# Al menos 1 autor con institución MX (en cualquier posición)
# =========================
has_mx_institution <- function(work) {
  auths <- work$authorships
  if (is.null(auths) || length(auths) == 0) return(FALSE)

  any(vapply(auths, function(a) {
    insts <- a$institutions
    if (is.null(insts) || length(insts) == 0) return(FALSE)
    cc <- vapply(insts, function(ii) ii$country_code %||% NA_character_, character(1))
    any(cc == "MX", na.rm = TRUE)
  }, logical(1)), na.rm = TRUE)
}

# safe_collapse que preserva "key:value" cuando llega named vector/list
safe_collapse_kv <- function(x) {
  if (is.null(x)) return(NA_character_)
  as.character(jsonlite::toJSON(x, auto_unbox = TRUE, null = "null"))
}

# =========================
# RECOMENDACIÓN: escritura por CHUNKS del JSON aplanado (evita crash por RAM)
# =========================
write_flat_csv_chunked <- function(all_results, termino, out_file, chunk_size = 200) {
  n <- length(all_results)
  idx <- split(seq_len(n), ceiling(seq_len(n) / chunk_size))

  # reiniciar archivo
  if (file.exists(out_file)) file.remove(out_file)

  for (b in seq_along(idx)) {
    ii <- idx[[b]]
    chunk <- all_results[ii]

    df_flat <- purrr::map_dfr(chunk, work_to_flat_row) %>%
      dplyr::mutate(search_term = termino, .before = 1)

    # append seguro (col_names solo en el primer chunk)
    readr::write_csv(
      df_flat,
      out_file,
      append = file.exists(out_file),
      col_names = !file.exists(out_file)
    )

    rm(df_flat, chunk)
    gc()
    cat(sprintf("   ✅ FULLFLAT chunk %d/%d escrito (%d filas)\n", b, length(idx), length(ii)))
  }
}

# =========================
# RECOMENDACIÓN: limpieza SOLO de columnas "humanas" (evita reventar RAM con JSON strings)
# =========================
limpiar_texto <- function(x) {
  x %>%
    tolower() %>%                                   # Convertir a minúsculas
    stringi::stri_trans_general(id = "Latin-ASCII") %>%      # Quitar tildes y acentos
    stringr::str_replace_all("<[^>]+>", " ") %>%             # Eliminar etiquetas HTML
    stringr::str_replace_all("&[a-z]+;", " ") %>%            # Eliminar entidades HTML tipo &nbsp;
    stringr::str_replace_all("[^[:print:]]", "") %>%         # Eliminar caracteres no imprimibles
    stringr::str_replace_all("[[:punct:]]+", " ") %>%        # Sustituir puntuación por espacio
    stringr::str_replace_all("\\s+", " ") %>%                # Reducir múltiples espacios
    stringr::str_trim()                                      # Quitar espacios al principio y al final
}

cols_texto_humano <- c("title","abstract","authors","institutions","countries","host_venue","display_name")

# =========================
# DEDUP GLOBAL (entre términos)
# =========================
seen_ids <- new.env(parent = emptyenv(), hash = TRUE)

is_new_global_id <- function(id) {
  if (is.na(id) || !nzchar(id)) return(FALSE)
  if (exists(id, envir = seen_ids, inherits = FALSE)) return(FALSE)
  assign(id, TRUE, envir = seen_ids)
  TRUE
}

# Bucle principal: repite el proceso para cada término de búsqueda.
for (termino in terminos) {
  cat("\n🔍 Buscando:", termino, "en México...\n")

  results_general <- list()                   # Lista vacía para guardar resultados
  query_general <- paste0('"', termino, '"')  # Arma la consulta con comillas
  filtro_pais <- ",authorships.countries:MX"  # Filtro para limitar a México
  cursor <- "*"; page <- 1                    # Variables para manejar la paginación

  # ===== reset meta/group_by por término (se llenará con la última página leída)
  meta_obj <- NULL
  group_by_obj <- NULL
  meta_count <- NA_integer_
  meta_page <- NA_integer_
  meta_per_page <- NA_integer_
  meta_next_cursor <- NA_character_
  group_by_json <- NA_character_

  # Este "repeat" se ejecuta hasta que no haya más páginas de resultados
  repeat {
    cat(paste0("   📄 Página ", page, "\n"))
    url <- paste0(
      "https://api.openalex.org/works?",
      "filter=title_and_abstract.search:", URLencode(query_general),
      filtro_pais, "&per-page=200&cursor=", URLencode(cursor)
    )

    resp <- try(httr::GET(url), silent = TRUE)                                 # Hace la solicitud al API
    if (inherits(resp, "try-error") || httr::status_code(resp) != 200) break
    data <- httr::content(resp, as = "parsed", type = "application/json")      # Convierte la respuesta a lista de R
    if (length(data$results) == 0) break                                       # Si no hay resultados, termina

    # ====== CORRECCIÓN: meta/group_by vienen de "data", NO de "resp"
    meta_obj <- data$meta %||% NULL
    group_by_obj <- data$group_by %||% NULL
    meta_count <- pick_int1(meta_obj$count)
    meta_page <- pick_int1(meta_obj$page)
    meta_per_page <- pick_int1(meta_obj$per_page)
    meta_next_cursor <- pick_chr1(meta_obj$next_cursor)
    group_by_json <- safe_json(group_by_obj)

    results_general <- append(results_general, data$results)  # Agrega resultados
    cursor <- data$meta$next_cursor                           # Pasa a la siguiente página
    if (is.null(cursor)) break
    page <- page + 1
    Sys.sleep(1)                                              # Espera un segundo (para no saturar el servidor)
    if (page > 300) break                                     # Límite de seguridad
  }

  cat("   → Registros obtenidos:", length(results_general), "\n")

  all_results <- results_general
  if (length(all_results) == 0) {
    cat("⚠️ Sin resultados para:", termino, "\n")
    terminos_sin_resultados <- c(terminos_sin_resultados, termino)
    next
  }

# ::::::::::::::::::::::::::::::::::::::::: DATAFRAME COMPLETO :::::::::::::::::::::::::::::::::::::::::
# Aquí se mapean (extraen) los datos específicos que interesan: autores, títulos, año, DOI, país, etc.
# Cada campo del API de OpenAlex se coloca como una columna en el dataframe "df".
# Tibble extendido (no quita columnas; solo agrega)
  df <- tibble::tibble(

    # =====================================================================
    # 1. IDENTIFICADORES BÁSICOS
    # =====================================================================
    search_term = termino,
    id = purrr::map_chr(all_results, ~ pick_chr1(.x$id)),
    doi = purrr::map_chr(all_results, ~ pick_chr1(.x$doi)),
    display_name = purrr::map_chr(all_results, ~ pick_chr1(.x$display_name)),
    title = purrr::map_chr(all_results, ~ pick_chr1(.x$title)),
    language = purrr::map_chr(all_results, ~ pick_chr1(.x$language)),

    # =====================================================================
    # 1B. URLS / API URLS (top-level)
    # =====================================================================
    url = purrr::map_chr(all_results, ~ pick_chr1(.x$url)),  # URL canónica del work
    cited_by_api_url = purrr::map_chr(all_results, ~ pick_chr1(.x$cited_by_api_url)),

    # =====================================================================
    # 1C. FLAGS IMPORTANTES (top-level)
    # =====================================================================
    is_authors_truncated = purrr::map_lgl(all_results, ~ pick_lgl1(.x$is_authors_truncated, default = FALSE)), # cuando authorships > 100

    # =====================================================================
    # 2. IDS {…}
    # =====================================================================
    ids_openalex = purrr::map_chr(all_results, ~ pick_chr1(.x$ids$openalex)),
    ids_doi = purrr::map_chr(all_results, ~ pick_chr1(.x$ids$doi)),
    ids_mag = purrr::map_chr(all_results, ~ pick_chr1(as.character(.x$ids$mag %||% NA_character_))),
    ids_pmid = purrr::map_chr(all_results, ~ pick_chr1(.x$ids$pmid)),
    ids_pmcid = purrr::map_chr(all_results, ~ pick_chr1(.x$ids$pmcid)),

    # =====================================================================
    # 3. PUBLICACIÓN
    # =====================================================================
    publication_year = purrr::map_int(all_results, ~ pick_int1(.x$publication_year)),
    publication_date = purrr::map_chr(all_results, ~ pick_chr1(.x$publication_date)),
    type = purrr::map_chr(all_results, ~ pick_chr1(.x$type)),
    type_crossref = purrr::map_chr(all_results, ~ pick_chr1(.x$type_crossref)),
    indexed_in = purrr::map_chr(all_results, ~ safe_collapse(.x$indexed_in)),

    # =====================================================================
    # 4. ABSTRACT
    # =====================================================================
    abstract = purrr::map_chr(all_results, function(x) {
      if (!is.null(x$abstract_inverted_index)) decode_abstract(x$abstract_inverted_index)
      else if (!is.null(x$abstract_inverted_index_v3)) decode_abstract(x$abstract_inverted_index_v3)
      else NA_character_
    }),

    # =====================================================================
    # 5. MÉTRICAS Y CITACIÓN
    # =====================================================================
    cited_by_count = purrr::map_int(all_results, ~ pick_int1(.x$cited_by_count %||% 0)),

    citation_normalized_percentile = purrr::map_chr(all_results, function(x) {
      val <- x$citation_normalized_percentile
      if (is.null(val)) return(NA_character_)
      if (length(val) == 1 && !is.list(val)) return(as.character(val))
      return(paste(val, collapse = "; "))
    }),
    citation_norm_pct_value = purrr::map_dbl(all_results, ~ pick_dbl1(.x$citation_normalized_percentile$value)),
    citation_norm_pct_top1 = purrr::map_lgl(all_results, ~ pick_lgl1(.x$citation_normalized_percentile$is_in_top_1_percent, default = FALSE)),
    citation_norm_pct_top10 = purrr::map_lgl(all_results, ~ pick_lgl1(.x$citation_normalized_percentile$is_in_top_10_percent, default = FALSE)),

    cited_by_percentile_year = purrr::map_chr(all_results, ~ paste(names(.x$cited_by_percentile_year %||% NA), collapse = "; ")),
    cited_by_percentile_year_min = purrr::map_dbl(all_results, ~ pick_dbl1(.x$cited_by_percentile_year$min)),
    cited_by_percentile_year_max = purrr::map_dbl(all_results, ~ pick_dbl1(.x$cited_by_percentile_year$max)),

    fwci = purrr::map_dbl(all_results, ~ pick_dbl1(.x$fwci)),
    relevance_score = purrr::map_dbl(all_results, ~ pick_dbl1(.x$relevance_score)),

    # =====================================================================
    # 6. OPEN ACCESS (UNA SOLA VEZ, sin duplicados)
    # =====================================================================
    is_oa = purrr::map_lgl(all_results, ~ pick_lgl1(.x$open_access$is_oa, default = FALSE)),
    open_access_status = purrr::map_chr(all_results, ~ pick_chr1(.x$open_access$oa_status)),
    open_access_url = purrr::map_chr(all_results, ~ pick_chr1(.x$open_access$oa_url)),
    open_access_version = purrr::map_chr(all_results, ~ pick_chr1(.x$open_access$version)),
    open_access_license = purrr::map_chr(all_results, ~ pick_chr1(.x$open_access$license)),
    open_access_repositories = purrr::map_chr(all_results, ~ pick_chr1(.x$open_access$repositories)),
    open_access_any_repository_has_fulltext = purrr::map_lgl(all_results, ~ pick_lgl1(.x$open_access$any_repository_has_fulltext, default = FALSE)),

    # =====================================================================
    # 7. BEST OA LOCATION
    # =====================================================================
    best_oa_location_url = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$landing_page_url)),
    best_oa_location_pdf_url = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$pdf_url)),
    best_oa_location_license = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$license)),
    best_oa_location_license_id = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$license_id)),
    best_oa_location_version = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$version)),
    best_oa_location_is_oa = purrr::map_lgl(all_results, ~ pick_lgl1(.x$best_oa_location$is_oa, default = FALSE)),
    best_oa_location_source_id = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$source$id)),
    best_oa_location_source_name = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$source$display_name)),
    best_oa_location_source_issn_l = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$source$issn_l)),
    best_oa_location_source_is_in_doaj = purrr::map_lgl(all_results, ~ pick_lgl1(.x$best_oa_location$source$is_in_doaj, default = FALSE)),
    best_oa_location_raw_source_name = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$raw_source_name)),
    best_oa_location_raw_type = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$raw_type)),

    # =====================================================================
    # 7B. BEST OA LOCATION (agregar campos faltantes del Location object)
    # =====================================================================
    best_oa_location_id = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$id)),
    best_oa_location_is_accepted = purrr::map_lgl(all_results, ~ pick_lgl1(.x$best_oa_location$is_accepted, default = FALSE)),
    best_oa_location_is_published = purrr::map_lgl(all_results, ~ pick_lgl1(.x$best_oa_location$is_published, default = FALSE)),

    best_oa_location_source_issn = purrr::map_chr(all_results, ~ safe_collapse(.x$best_oa_location$source$issn)),
    best_oa_location_source_is_oa = purrr::map_lgl(all_results, ~ pick_lgl1(.x$best_oa_location$source$is_oa, default = FALSE)),
    best_oa_location_source_is_core = purrr::map_lgl(all_results, ~ pick_lgl1(.x$best_oa_location$source$is_core, default = FALSE)),
    best_oa_location_source_host_organization = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$source$host_organization)),
    best_oa_location_source_host_organization_name = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$source$host_organization_name)),
    best_oa_location_source_host_organization_lineage = purrr::map_chr(all_results, ~ safe_collapse(.x$best_oa_location$source$host_organization_lineage)),
    best_oa_location_source_host_organization_lineage_names = purrr::map_chr(all_results, ~ safe_collapse(.x$best_oa_location$source$host_organization_lineage_names)),
    best_oa_location_source_type = purrr::map_chr(all_results, ~ pick_chr1(.x$best_oa_location$source$type)),

    # =====================================================================
    # 8. PRIMARY LOCATION (UNIFICADO, SIN REPETIDOS)
    # =====================================================================
    primary_location_landing_page_url = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$landing_page_url)),
    primary_location_pdf_url          = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$pdf_url)),
    primary_location_license          = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$license)),
    primary_location_license_id       = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$license_id)),
    primary_location_version          = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$version)),
    primary_location_is_oa            = purrr::map_lgl(all_results, ~ pick_lgl1(.x$primary_location$is_oa,        default = FALSE)),
    primary_location_is_accepted      = purrr::map_lgl(all_results, ~ pick_lgl1(.x$primary_location$is_accepted,  default = FALSE)),
    primary_location_is_published     = purrr::map_lgl(all_results, ~ pick_lgl1(.x$primary_location$is_published, default = FALSE)),

    primary_location_id               = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$id)),

    # ---- source (todo en una sola familia de nombres coherente)
    primary_location_source_id        = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$source$id)),
    primary_location_source_name      = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$source$display_name)),
    primary_location_source_issn_l    = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$source$issn_l)),
    primary_location_source_issn      = purrr::map_chr(all_results, ~ safe_collapse(.x$primary_location$source$issn)),

    primary_location_source_is_oa      = purrr::map_lgl(all_results, ~ pick_lgl1(.x$primary_location$source$is_oa,      default = FALSE)),
    primary_location_source_is_in_doaj = purrr::map_lgl(all_results, ~ pick_lgl1(.x$primary_location$source$is_in_doaj, default = FALSE)),
    primary_location_source_is_core    = purrr::map_lgl(all_results, ~ pick_lgl1(.x$primary_location$source$is_core,    default = FALSE)),

    primary_location_source_host_organization       = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$source$host_organization)),
    primary_location_source_host_organization_name  = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$source$host_organization_name)),
    primary_location_source_host_organization_lineage       = purrr::map_chr(all_results, ~ safe_collapse(.x$primary_location$source$host_organization_lineage)),
    primary_location_source_host_organization_lineage_names = purrr::map_chr(all_results, ~ safe_collapse(.x$primary_location$source$host_organization_lineage_names)),

    primary_location_source_type      = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$source$type)),

    # ---- raw
    raw_source_name                   = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$raw_source_name)),
    raw_type                          = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_location$raw_type)),

    # =====================================================================
    # 9. LOCATIONS []
    # =====================================================================
    locations_count = purrr::map_int(all_results, ~ pick_int1(.x$locations_count %||% 0)),
    locations_landing_page_urls = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, "landing_page_url"))),
    locations_pdf_urls = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, "pdf_url"))),
    locations_versions = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, "version"))),
    locations_licenses = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, "license"))),
    locations_license_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, "license_id"))),
    locations_is_oa = purrr::map_chr(all_results, ~ safe_collapse(purrr::map_chr(.x$locations, ~ as.character(.x$is_oa %||% FALSE)))),
    locations_source_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "id")))),
    locations_source_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "display_name")))),
    locations_source_issn_l = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "issn_l")))),
    locations_source_is_in_doaj = purrr::map_chr(all_results, ~ safe_collapse(purrr::map_chr(.x$locations, ~ as.character(.x$source$is_in_doaj %||% FALSE)))),
    locations_raw_source_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, "raw_source_name"))),
    locations_raw_types = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, "raw_type"))),

    # =====================================================================
    # 9B. LOCATIONS [] (agregar campos faltantes del Location object, colapsados)
    # =====================================================================
    locations_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, "id"))),
    locations_is_accepted = purrr::map_chr(all_results, ~ safe_collapse(purrr::map_chr(.x$locations, ~ as.character(.x$is_accepted %||% FALSE)))),
    locations_is_published = purrr::map_chr(all_results, ~ safe_collapse(purrr::map_chr(.x$locations, ~ as.character(.x$is_published %||% FALSE)))),

    locations_source_issn = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "issn")))),
    locations_source_is_oa = purrr::map_chr(all_results, ~ safe_collapse(purrr::map_chr(.x$locations, ~ as.character(.x$source$is_oa %||% FALSE)))),
    locations_source_is_core = purrr::map_chr(all_results, ~ safe_collapse(purrr::map_chr(.x$locations, ~ as.character(.x$source$is_core %||% FALSE)))),
    locations_source_host_organization = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "host_organization")))),
    locations_source_host_organization_name = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "host_organization_name")))),
    locations_source_host_organization_lineage = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "host_organization_lineage")))),
    locations_source_host_organization_lineage_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "host_organization_lineage_names")))),
    locations_source_types = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$locations, c("source", "type")))),

    # =====================================================================
    # 10. HOST VENUE
    # =====================================================================
    host_venue = purrr::map_chr(all_results, ~ pick_chr1(.x$host_venue$display_name)),
    issn = purrr::map_chr(all_results, ~ pick_chr1(paste(.x$host_venue$issn %||% NA_character_, collapse = ", "))),
    license = purrr::map_chr(all_results, ~ pick_chr1(.x$host_venue$license)),

    # =====================================================================
    # 10. HOST VENUE (DEPRECADO)
    # OJO: Tu sesión ha fallado porque se ha usado toda la memoria RAM disponible.ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$authorships, c("author", "id")))),
    author_alt_names = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, ~ .x$author$display_name_alternatives)))),
    author_orcids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$authorships, c("author", "orcid")))),
    author_positions = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$authorships, "author_position"))),
    authors_is_corresponding = purrr::map_chr(all_results, ~ safe_collapse(purrr::map_chr(.x$authorships, ~ as.character(.x$is_corresponding)))),
    raw_author_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$authorships, "raw_author_name"))),
    raw_affiliation_strings = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, "raw_affiliation_strings")))),
    affiliations_institution_ids = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, ~ unlist(purrr::map(.x$affiliations, "institution_ids")))))),

    institutions = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, ~ purrr::map(.x$institutions, "display_name"))))),
    institution_ids = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, ~ purrr::map(.x$institutions, "id"))))),
    institution_rors = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, ~ purrr::map(.x$institutions, "ror"))))),
    institution_country_codes = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, ~ purrr::map(.x$institutions, "country_code"))))),
    institution_types = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, ~ purrr::map(.x$institutions, "type"))))),
    institution_lineage = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, ~ purrr::map(.x$institutions, "lineage"))))),

    countries = purrr::map_chr(all_results, ~ safe_collapse(unlist(purrr::map(.x$authorships, "countries")))),
    countries_distinct_count = purrr::map_int(all_results, ~ pick_int1(.x$countries_distinct_count %||% 0)),
    institutions_distinct_count = purrr::map_int(all_results, ~ pick_int1(.x$institutions_distinct_count %||% 0)),
    corresponding_author_ids = purrr::map_chr(all_results, ~ safe_collapse(.x$corresponding_author_ids)),
    corresponding_institution_ids = purrr::map_chr(all_results, ~ safe_collapse(.x$corresponding_institution_ids)),

    # =====================================================================
    # 12. BIBLIO
    # =====================================================================
    volume = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$volume)),
    issue = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$issue)),
    first_page = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$first_page)),
    last_page = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$last_page)),
    article_number = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$article_number)),

    # =====================================================================
    # 12B. BIBLIO (asegurar campos + "article_number" si aparece)
    # =====================================================================
    biblio_volume = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$volume)),
    biblio_issue = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$issue)),
    biblio_first_page = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$first_page)),
    biblio_last_page = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$last_page)),
    biblio_article_number = purrr::map_chr(all_results, ~ pick_chr1(.x$biblio$article_number)),

    # =====================================================================
    # 13. TOPICS / CONCEPTS / KEYWORDS / MESH / SDG
    # =====================================================================
    primary_topic = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_topic$display_name)),
    primary_topic_id = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_topic$id)),
    primary_topic_score = purrr::map_dbl(all_results, ~ pick_dbl1(.x$primary_topic$score)),
    primary_topic_subfield_id = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_topic$subfield$id)),
    primary_topic_subfield_name = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_topic$subfield$display_name)),
    primary_topic_field_id = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_topic$field$id)),
    primary_topic_field_name = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_topic$field$display_name)),
    primary_topic_domain_id = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_topic$domain$id)),
    primary_topic_domain_name = purrr::map_chr(all_results, ~ pick_chr1(.x$primary_topic$domain$display_name)),

    topics = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, "display_name"))),
    topics_scores = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, "score"))),
    topics_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, "id"))),
    topics_subfield_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, c("subfield", "id")))),
    topics_subfield_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, c("subfield", "display_name")))),
    topics_field_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, c("field", "id")))),
    topics_field_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, c("field", "display_name")))),
    topics_domain_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, c("domain", "id")))),
    topics_domain_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$topics, c("domain", "display_name")))),

    keywords = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$keywords, "display_name"))),
    keywords_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$keywords, "id"))),
    keywords_scores = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$keywords, "score"))),

    mesh_terms = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$mesh, "descriptor_name"))),
    mesh_descriptor_ui = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$mesh, "descriptor_ui"))),
    mesh_is_major_topic = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$mesh, ~ as.character(.x$is_major_topic)))),

    concepts = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$concepts, "display_name"))),
    concepts_scores = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$concepts, "score"))),
    concepts_levels = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$concepts, "level"))),
    concepts_wikidata = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$concepts, "wikidata"))),

    sdg = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$sustainable_development_goals, "display_name"))),
    sdg_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$sustainable_development_goals, "id"))),
    sdg_scores = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$sustainable_development_goals, "score"))),

    # =====================================================================
    # 14. REFERENCIAS
    # =====================================================================
    referenced_works_count = purrr::map_int(all_results, ~ pick_int1(.x$referenced_works_count %||% 0)),
    referenced_works = purrr::map_chr(all_results, ~ safe_collapse(.x$referenced_works)),
    related_works = purrr::map_chr(all_results, ~ safe_collapse(.x$related_works)),

    # =====================================================================
    # 15. DATASETS Y VERSIONES
    # =====================================================================
    datasets = purrr::map_chr(all_results, ~ safe_collapse(.x$datasets)),
    versions = purrr::map_chr(all_results, ~ safe_collapse(.x$versions)),

    # =====================================================================
    # 16. APC / FINANCIAMIENTO / PREMIOS
    # =====================================================================
    apc_paid = purrr::map_chr(all_results, function(x) {
      if (is.null(x$apc_paid)) return(NA_character_)
      if (is.numeric(x$apc_paid)) return(as.character(x$apc_paid))
      if (is.list(x$apc_paid)) return(paste(unlist(x$apc_paid), collapse = "; "))
      NA_character_
    }),
    apc_list = purrr::map_chr(all_results, ~ safe_collapse(.x$apc_list)),
    grants = purrr::map_chr(all_results, function(x) {
      if (is.null(x$grants)) return(NA_character_)
      safe_collapse(purrr::map_chr(x$grants, ~ paste0(.x$funder_display_name, " (", .x$grant_id, ")")))
    }),
    funders = purrr::map_chr(all_results, ~ safe_collapse(.x$funders)),
    awards = purrr::map_chr(all_results, ~ safe_collapse(.x$awards)),

    # =====================================================================
    # 16B. APC (aplanado completo de apc_list y apc_paid)
    # =====================================================================
    apc_list_value = purrr::map_int(all_results, ~ pick_int1(.x$apc_list$value)),
    apc_list_currency = purrr::map_chr(all_results, ~ pick_chr1(.x$apc_list$currency)),
    apc_list_provenance = purrr::map_chr(all_results, ~ pick_chr1(.x$apc_list$provenance)),
    apc_list_value_usd = purrr::map_int(all_results, ~ pick_int1(.x$apc_list$value_usd)),

    apc_paid_value = purrr::map_int(all_results, ~ pick_int1(.x$apc_paid$value)),
    apc_paid_currency = purrr::map_chr(all_results, ~ pick_chr1(.x$apc_paid$currency)),
    apc_paid_provenance = purrr::map_chr(all_results, ~ pick_chr1(.x$apc_paid$provenance)),
    apc_paid_value_usd = purrr::map_int(all_results, ~ pick_int1(.x$apc_paid$value_usd)),

    # =====================================================================
    # 16C. FUNDERS / AWARDS (evitar "[object Object]" -> sacar id y display_name)
    # =====================================================================
    funders_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$funders, "id"))),
    funders_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$funders, "display_name"))),

    awards_ids = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$awards, "id"))),
    awards_display_names = purrr::map_chr(all_results, ~ safe_collapse(purrr::map(.x$awards, "display_name"))),

    # =====================================================================
    # 17. COUNTS BY YEAR
    # =====================================================================
    counts_by_year = purrr::map_chr(all_results, function(x) {
      if (is.null(x$counts_by_year)) return(NA_character_)
      paste(sapply(x$counts_by_year, function(c) paste0(c$year, ":", c$cited_by_count)), collapse = "; ")
    }),
    counts_by_year_kv = purrr::map_chr(all_results, ~ safe_collapse_kv(.x$counts_by_year)),

    # =====================================================================
    # 18. CONTENIDO Y FULL-TEXT (sin duplicados + FIX content_url)
    # =====================================================================
    has_fulltext = purrr::map_lgl(all_results, ~ pick_lgl1(.x$has_fulltext, default = FALSE)),
    is_paratext = purrr::map_lgl(all_results, ~ pick_lgl1(.x$is_paratext, default = FALSE)),
    is_retracted = purrr::map_lgl(all_results, ~ pick_lgl1(.x$is_retracted, default = FALSE)),
    is_xpac = purrr::map_lgl(all_results, ~ pick_lgl1(.x$is_xpac, default = FALSE)),

    # has_content suele ser objeto con llaves (pdf, grobid_xml). No lo trates como scalar lógico “plano”.
    has_content_json = purrr::map_chr(all_results, ~ safe_json(.x$has_content)),
    has_content_pdf = purrr::map_lgl(all_results, ~ pick_lgl1(.x$has_content$pdf, default = FALSE)),
    has_content_grobid_xml = purrr::map_lgl(all_results, ~ pick_lgl1(.x$has_content$grobid_xml, default = FALSE)),

    # content_url A VECES NO ES length-1 -> pick_chr1 lo hace seguro
    content_url = purrr::map_chr(all_results, ~ pick_chr1(.x$content_url)),

    # content_urls (plural) es objeto nombrado -> JSON o key:value
    content_urls_json = purrr::map_chr(all_results, ~ safe_json(.x$content_urls)),

    # =====================================================================
    # 19. TEMPORALIDAD
    # =====================================================================
    created_date = purrr::map_chr(all_results, ~ pick_chr1(.x$created_date)),
    updated_date = purrr::map_chr(all_results, ~ pick_chr1(.x$updated_date)),

    # =====================================================================
    # 20. RAW JSON Y META/GROUP_BY (del endpoint, replicados)
    # =====================================================================
    raw_json = purrr::map_chr(all_results, ~ jsonlite::toJSON(.x, auto_unbox = TRUE, null = "null")),
    meta_count = meta_count,
    meta_page = meta_page,
    meta_per_page = meta_per_page,
    meta_next_cursor = meta_next_cursor,
    group_by = group_by_json,
    db_source = "OpenAlex"
  )

# =========================
# Volcado completo + combinación (DENTRO del for)
# =========================
df_curada <- df

# ===== RECOMENDACIÓN CRÍTICA: NO map_dfr gigante (revienta RAM). Haz FULLFLAT por chunks a disco.
fullflat_file <- paste0("/content/por_termino_covid_mx/",
                        gsub("[^A-Za-z0-9]+", "_", tolower(termino)),
                        "_OpenAlex_MX_FULLFLAT.csv")

write_flat_csv_chunked(all_results, termino, fullflat_file, chunk_size = 200)

# En lugar de unir df_curada + df_full_flat (gigante), nos quedamos con df_curada + raw_json (ya viene en df)
# Si aún quieres "combinar", lo haces offline leyendo por partes; aquí NO lo hacemos para no crashear.

# A partir de aquí, limpia y guarda df (curado + raw_json) (no FULLFLAT)
  # ::::::::::::::::::::::::::::::::::::::::: LIMPIEZA ROBUSTA :::::::::::::::::::::::::::::::::::::::::

  library(stringi)

  # Aquí se limpian los textos para quitar símbolos raros, tildes, mayúsculas, HTML, etc.
  # Reemplazar cadenas vacías por NA
  df[df == ""] <- NA

  # Aplicar limpieza general a columnas "humanas" (NO a todas, para no destrozar RAM con JSON)
  df <- df %>%
    mutate(across(where(~ is.character(.x) && !inherits(.x, "json")), ~ na_if(., "NA"))) %>%
    mutate(across(any_of(cols_texto_humano), limpiar_texto))

  # Luego se filtra para quedarse solo con publicaciones asociadas explícitamente a México.
  # Filtrar solo registros que contienen explícitamente 'mx' como país (y no mezclado con otros)
  df <- df %>%
    filter(!is.na(countries)) %>%
    filter(str_detect(countries, "\\bmexico\\b|\\bmx\\b"))

  # --------- filtro fuerte: ≥1 autor con institución MX (cualquier posición)
  mx_inst_flag <- purrr::map_lgl(all_results, has_mx_institution)

  # ojo: mx_inst_flag está alineado con all_results, pero df proviene de all_results
  # como df y all_results están 1:1 en este punto, lo añadimos como columna
  df$has_mx_institution <- mx_inst_flag

  df <- df %>%
    filter(has_mx_institution)

  # --------- dedup global por ID OpenAlex (entre términos)
  keep <- vapply(df$id, is_new_global_id, logical(1))
  df <- df[keep, , drop = FALSE]

  # (opcional) dedup local adicional por si acaso
  df <- df %>% distinct(id, .keep_all = TRUE)

  # ::::::::::::::::::::::::::::::::::::::::: GUARDAR CSV :::::::::::::::::::::::::::::::::::::::::
  # Se guarda un archivo CSV con los resultados de cada término.
  filename <- paste0("/content/por_termino_covid_mx/", gsub("[^A-Za-z0-9]+", "_", tolower(termino)), "_OpenAlex_MX.csv")
  if (nrow(df) > 0) {
    write_csv(df, filename)
    cat("✅ Archivo guardado:", filename, "con", nrow(df), "registros\n")
    cat("✅ Archivo FULLFLAT (por chunks) guardado:", fullflat_file, "\n")
    terminos_con_resultados <- c(terminos_con_resultados, termino)
  } else {
    cat("⚠️ No se generó archivo para:", termino, "(sin registros válidos)\n")
    terminos_sin_resultados <- c(terminos_sin_resultados, termino)
  }

  # ===== RECOMENDACIÓN: liberar memoria por término (evita crash acumulativo)
  rm(all_results, results_general, data)
  gc()
}

# ::::::::::::::::::::::::::::::::::::::::: CREAR ZIP ::::::::::::::::::::::::::::::::::::::::::::::::::::::
# Al final, se comprimen todos los archivos CSV en un solo ZIP para facilitar la descarga.
csv_files <- dir("/content/por_termino_covid_mx", pattern = "\\.csv$", full.names = TRUE)
csv_files <- csv_files[file.exists(csv_files)]
cat("🔢 Total de archivos CSV a comprimir:", length(csv_files), "\n")

zip::zipr(zipfile = "/content/resultados_COVID_MX.zip", files = csv_files)
cat("📦 ZIP creado correctamente con", length(csv_files), "archivos\n")

# ::::::::::::::::::::::::::::::::::::::::: LOG ::::::::::::::::::::::::::::::::::::::::::::::::::::::
# Se crea un archivo de registro (log) con un resumen de lo que se buscó y cuántos resultados hubo por término.
log_df <- tibble(
  term = terminos,
  archivo = sapply(terminos, function(t) paste0("/content/por_termino_covid_mx/", gsub("[^A-Za-z0-9]+", "_", tolower(t)), "_OpenAlex_MX.csv")),
  n_resultados = sapply(terminos, function(t) {
    archivo <- paste0("/content/por_termino_covid_mx/", gsub("[^A-Za-z0-9]+", "_", tolower(t)), "_OpenAlex_MX.csv")
    if (file.exists(archivo)) nrow(read_csv(archivo, show_col_types = FALSE)) else 0
  })
) %>%
  mutate(found_results = n_resultados > 0)

write_csv(log_df, "/content/log_busqueda_COVID_MX.csv")
write_csv(df, filename)

cat("\n📋 Resumen general:\n")
cat("✅ Términos con resultados:", paste(terminos_con_resultados, collapse = ", "), "\n")
cat("⚠️ Términos sin resultados:", paste(terminos_sin_resultados, collapse = ", "), "\n")
cat("🧾 Log guardado en: /content/log_busqueda_COVID_MX.csv\n")



🔍 Buscando: COVID-19 en México...
   📄 Página 1
   📄 Página 2
   📄 Página 3
   📄 Página 4
   📄 Página 5
   📄 Página 6
   📄 Página 7
   📄 Página 8
   📄 Página 9
   📄 Página 10
   📄 Página 11
   📄 Página 12
   📄 Página 13
   📄 Página 14
   📄 Página 15
   📄 Página 16
   📄 Página 17
   📄 Página 18
   📄 Página 19
   📄 Página 20
   📄 Página 21
   📄 Página 22
   📄 Página 23
   📄 Página 24
   📄 Página 25
   📄 Página 26
   📄 Página 27
   📄 Página 28
   📄 Página 29
   📄 Página 30
   📄 Página 31
   📄 Página 32
   📄 Página 33
   📄 Página 34
   📄 Página 35
   📄 Página 36
   📄 Página 37
   📄 Página 38
   📄 Página 39
   📄 Página 40
   📄 Página 41
   📄 Página 42
   📄 Página 43
   📄 Página 44
   📄 Página 45
   📄 Página 46
   📄 Página 47
   📄 Página 48
   📄 Página 49
   📄 Página 50
   📄 Página 51
   📄 Página 52
   📄 Página 53
   📄 Página 54
   📄 Página 55
   📄 Página 56
   📄 Página 57
   📄 Página 58
   📄 Página 59
   📄 Página 60
   📄 Página 61
   📄 Página 62
   📄 Página 63
   📄 Página 64
   📄 Página 65

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”



📋 Resumen general:
✅ Términos con resultados: COVID-19, COVID19, Coronavirus, Corona virus, 2019-nCoV, SARS-CoV2, SARS-CoV-2, SARS-CoV, Severe Acute Respiratory Syndrome, Middle East Respiratory Syndrome 
⚠️ Términos sin resultados: MERS-CoV 
🧾 Log guardado en: /content/log_busqueda_COVID_MX.csv


# Explicación detallada del script de consolidación OpenAlex (Curado + FULLFLAT a Parquet)

Este script en **R** consolida los archivos generados previamente por término de búsqueda en OpenAlex y ejecuta dos procesos paralelos claramente diferenciados:

1. **Unificación y deduplicación de archivos curados (CSV).**
2. **Conversión de archivos FULLFLAT a formato Parquet sin consumir excesiva memoria.**

Además, implementa:

- Auditoría robusta de parsing.
- Fallback automático de encoding.
- Registro de duplicados.
- Índice global por `id`.
- Manifest estructurado de archivos procesados.
- Logs técnicos de problemas de lectura.

El objetivo es transformar múltiples archivos por término en una estructura coherente, eficiente y escalable.

---

# 1) Carga de librerías y configuración

Se cargan silenciosamente los paquetes necesarios:

- readr
- dplyr
- purrr
- stringr
- tibble

Para la conversión FULLFLAT → Parquet se instala y carga `arrow`.

Esto permite:

- Trabajar con CSV de manera estable.
- Procesar grandes volúmenes sin saturar RAM.
- Escribir datasets en formato columnar optimizado (Parquet).

---

# 2) Definición de rutas y parámetros

## 2.1 Directorio de entrada

`/content/por_termino_covid_mx`

Aquí se espera encontrar:

- Archivos curados (`*_OpenAlex_MX.csv`)
- Archivos FULLFLAT (`*_OpenAlex_MX_FULLFLAT.csv`)

## 2.2 Salidas principales

### Curado unificado
- OA_COVID19_MX_curado_unificado.csv
- OA_COVID19_MX_curado_duplicados.csv
- log_OA_COVID19_MX_curado_unificado.csv

### FULLFLAT en Parquet
- Carpeta: `/content/fullflat_parquet`
- Índice de ids: `fullflat_index_ids.csv`
- Manifest: `fullflat_manifest.csv`

### Logs técnicos
- `log_parsing_issues_unificado_dos_carriles.csv`

## 2.3 Preferencias

- Encoding preferido: UTF-8
- Fallback automático a Latin1
- Opción para silenciar warnings en consola
- Opción para guardar log de parsing

---

# 3) Listado y clasificación de archivos

El script:

1. Lista todos los CSV en el directorio.
2. Separa en dos grupos:

   - Curados (sin FULLFLAT en el nombre)
   - FULLFLAT (terminan en `_FULLFLAT.csv`)

Esto permite ejecutar dos pipelines independientes.

---

# 4) Lector robusto compatible con múltiples versiones de readr

Uno de los puntos más técnicos del script es el lector robusto.

Problema que resuelve:
Distintas versiones de `readr` soportan distintos argumentos (por ejemplo, `escape_backslash`, `lazy`, etc.).

Solución implementada:

- Se inspeccionan los argumentos reales disponibles en la versión instalada.
- Solo se pasan a `read_csv` los argumentos que existen.
- Se intenta primero con UTF-8.
- Si falla, se reintenta con Latin1.
- Se registran los problemas detectados por `problems()`.

Esto evita errores como:

- "unused argument escape_backslash"
- Fallos silenciosos por encoding

El resultado es una lectura adaptable a entornos heterogéneos.

---

# 5) Pipeline 1: Unificación de archivos curados

## 5.1 Normalización del identificador

Se crea una columna `id_limpio` que:

- Convierte a minúsculas
- Elimina espacios
- Reemplaza cadenas vacías por NA

Esto garantiza consistencia para deduplicación.

## 5.2 Unión de archivos

Todos los curados se combinan con `bind_rows()`.

Se eliminan filas sin identificador válido.

## 5.3 Detección de duplicados

Se marca duplicación usando:

`duplicated(df_cur$id_limpio)`

Se generan dos datasets:

- Únicos
- Duplicados

## 5.4 Archivos generados

- CSV unificado sin duplicados.
- CSV con registros duplicados.
- Log resumen con:

  - Número de archivos
  - Total de registros
  - Registros únicos
  - Registros duplicados
  - Porcentaje de duplicación

Este paso consolida el dataset listo para análisis bibliométrico.

---

# 6) Pipeline 2: Conversión FULLFLAT a Parquet

El FULLFLAT puede ser extremadamente grande.  
Para evitar colapsos de memoria:

1. Se procesa archivo por archivo.
2. Cada archivo se lee con el lector robusto.
3. Se genera `id_limpio`.
4. Se construye un índice de ids únicos.
5. Se escribe directamente en formato Parquet.

No se concatenan todos los FULLFLAT en RAM.

## 6.1 Índice global

Se genera un archivo:

`fullflat_index_ids.csv`

Contiene:

- id_limpio
- archivo de origen

Esto permite búsquedas rápidas sin reabrir todos los datasets.

## 6.2 Manifest técnico

Se genera:

`fullflat_manifest.csv`

Incluye por archivo:

- Ruta CSV original
- Ruta Parquet
- Número total de filas
- Número con id válido
- Número de columnas

Este manifest actúa como inventario estructural.

---

# 7) Registro de problemas de parsing

Si durante la lectura se detectan inconsistencias (por ejemplo columnas mal formateadas):

- Se almacenan en `parsing_issues_log`
- Se ordenan por archivo, fila y columna
- Se exportan a CSV

Si no se detectan problemas, el script lo indica explícitamente.

Esto es importante para auditoría y reproducibilidad.

---

# 8) Resultados esperados

Al finalizar la ejecución, se obtendrá:

## Curado
- OA_COVID19_MX_curado_unificado.csv
- OA_COVID19_MX_curado_duplicados.csv
- log_OA_COVID19_MX_curado_unificado.csv

## FULLFLAT optimizado
- Carpeta `/content/fullflat_parquet`
- fullflat_index_ids.csv
- fullflat_manifest.csv

## Logs técnicos
- log_parsing_issues_unificado_dos_carriles.csv (si aplica)

---

# Arquitectura metodológica del script

Este programa implementa un modelo de "dos carriles":

Carril 1 → Curado consolidado en CSV (listo para análisis directo).  
Carril 2 → FULLFLAT convertido a dataset Parquet escalable.

Ventajas del diseño:

- Separación clara entre dataset analítico y dataset estructural.
- Control explícito de memoria.
- Adaptabilidad a múltiples versiones de readr.
- Registro exhaustivo de duplicación y parsing.
- Escalabilidad futura mediante Parquet.

---

# Garantías del script

Garantiza:

- Deduplicación consistente por identificador.
- Compatibilidad inter-versiones de readr.
- Conservación íntegra del FULLFLAT.
- Índice estructurado para recuperación rápida.
- Auditoría técnica documentada.

No garantiza:

- Corrección semántica del contenido original.
- Eliminación de duplicados conceptuales (solo por id).
- Que los parsing issues sean irrelevantes; deben revisarse si aparecen.

---

# Resultado final conceptual

El script transforma múltiples salidas por término en:

- Un dataset analítico limpio y consolidado.
- Un backend estructural optimizado en formato columnar.
- Un sistema documentado de control de calidad.

En términos prácticos:  
El CSV unificado es el instrumento de análisis.  
El Parquet FULLFLAT es la base estructural de respaldo.



In [ ]:
# =========================================================
# CURADO -> CSV unificado (OK en RAM)
# FULLFLAT -> Parquet por archivo (NO revienta RAM)
# + logs + índice por id
# + LECTOR ROBUSTO (compatible con tu versión de readr)
# =========================================================

suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(purrr)
  library(stringr)
  library(tibble)
})

# ---- (para FULLFLAT->Parquet) instalar/cargar arrow
if (!requireNamespace("arrow", quietly = TRUE)) {
  install.packages("arrow", repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(library(arrow))

# -------------------------
# RUTAS
# -------------------------
DIR_IN <- "/content/por_termino_covid_mx"

OUT_CURADO_UNI      <- "/content/OA_COVID19_MX_curado_unificado.csv"
OUT_CURADO_DUPS     <- "/content/OA_COVID19_MX_curado_duplicados.csv"
OUT_LOG_CURADO      <- "/content/log_OA_COVID19_MX_curado_unificado.csv"

OUT_FULLFLAT_DIR    <- "/content/fullflat_parquet"
OUT_FULLFLAT_INDEX  <- "/content/fullflat_index_ids.csv"
OUT_FULLFLAT_MANIF  <- "/content/fullflat_manifest.csv"

OUT_LOG_PARSING     <- "/content/log_parsing_issues_unificado_dos_carriles.csv"
GUARDAR_LOG_PARSING <- TRUE

# Preferencias
ENCODING_PREFERIDO <- "UTF-8"   # fallback automático a Latin1 si falla
SILENCIAR_WARNINGS_EN_PANTALLA <- TRUE

dir.create(OUT_FULLFLAT_DIR, showWarnings = FALSE, recursive = TRUE)

# -------------------------
# 1) LISTAR ARCHIVOS
# -------------------------
all_csv <- list.files(DIR_IN, pattern = "\\.csv$", full.names = TRUE)
if (length(all_csv) == 0) stop("No hay CSVs en: ", DIR_IN)

curados <- all_csv[
  grepl("_OpenAlex_MX\\.csv$", all_csv, ignore.case = TRUE) &
    !grepl("FULLFLAT", all_csv, ignore.case = TRUE)
]
fullflats <- all_csv[grepl("_OpenAlex_MX_FULLFLAT\\.csv$", all_csv, ignore.case = TRUE)]

cat("📄 Curados encontrados:    ", length(curados), "\n")
cat("📄 FULLFLAT encontrados:   ", length(fullflats), "\n")

# -------------------------
# 2) Auditoría de parsing + lector robusto (compat)
# -------------------------
parsing_issues_log <- list()

# helper: ¿read_csv soporta arg X?
read_csv_formals <- names(formals(readr::read_csv))
has_arg <- function(arg) arg %in% read_csv_formals

read_csv_robusto <- function(f, encoding) {
  # armamos args base
  args <- list(
    file = f,
    col_types = cols(.default = "c"),
    show_col_types = FALSE,
    progress = FALSE,
    locale = readr::locale(encoding = encoding)
  )

  # lazy=FALSE (si existe) suele ser más estable con CSVs raros
  if (has_arg("lazy")) args$lazy <- FALSE

  # quote (si existe)
  if (has_arg("quote")) args$quote <- "\""

  # OJO: estos args NO existen en tu versión, por eso los ponemos solo si existen
  if (has_arg("escape_backslash")) args$escape_backslash <- TRUE
  if (has_arg("escape_double"))    args$escape_double    <- TRUE

  # ejecutar con do.call para pasar solo lo soportado
  do.call(readr::read_csv, args)
}

read_csv_audit <- function(f) {

  read_fun <- function() {
    # intento 1: encoding preferido
    dat <- tryCatch(
      read_csv_robusto(f, ENCODING_PREFERIDO),
      error = function(e) e
    )

    # fallback: Latin1 si el primer intento falla
    if (inherits(dat, "error")) {
      dat <- read_csv_robusto(f, "Latin1")
    }

    pr <- readr::problems(dat)
    if (GUARDAR_LOG_PARSING && nrow(pr) > 0) {
      pr$file <- f
      parsing_issues_log[[length(parsing_issues_log) + 1]] <<- pr
    }
    dat
  }

  if (SILENCIAR_WARNINGS_EN_PANTALLA) suppressWarnings(read_fun()) else read_fun()
}

# -------------------------
# 3) CURADOS: unificar y deduplicar por id
# -------------------------
add_id_limpio <- function(df) {
  if (!("id" %in% names(df))) df$id <- NA_character_
  df %>%
    mutate(
      id_limpio = tolower(trimws(id)),
      id_limpio = na_if(id_limpio, ""),
      id_limpio = na_if(id_limpio, "na")
    )
}

if (length(curados) > 0) {
  cur_list <- map(curados, read_csv_audit)

  df_cur <- bind_rows(cur_list) %>%
    add_id_limpio() %>%
    filter(!is.na(id_limpio))

  total_cur <- nrow(df_cur)
  dup_cur <- duplicated(df_cur$id_limpio)

  df_cur_unicos <- df_cur[!dup_cur, ]
  df_cur_dups   <- df_cur[ dup_cur, ]

  write_csv(df_cur_unicos, OUT_CURADO_UNI)
  write_csv(df_cur_dups,   OUT_CURADO_DUPS)

  log_cur <- tibble(
    tipo = "curado",
    archivos = length(curados),
    total_registros = total_cur,
    registros_unicos = nrow(df_cur_unicos),
    registros_duplicados = nrow(df_cur_dups),
    porcentaje_duplicados = round(nrow(df_cur_dups) / max(total_cur, 1) * 100, 2)
  )
  write_csv(log_cur, OUT_LOG_CURADO)

  cat("✅ Curado unificado:", OUT_CURADO_UNI, "(", nrow(df_cur_unicos), ")\n")
  cat("🧾 Curado duplicados:", OUT_CURADO_DUPS, "(", nrow(df_cur_dups), ")\n")
} else {
  df_cur_unicos <- tibble()
}

rm(cur_list); invisible(gc())

# -------------------------
# 4) FULLFLAT: CSV -> Parquet por archivo + índice ids
# -------------------------
fullflat_index <- list()
fullflat_manifest <- list()

if (length(fullflats) > 0) {

  for (f in fullflats) {
    cat("\n🧱 FULLFLAT -> Parquet:", basename(f), "\n")

    dat <- read_csv_audit(f)

    if (!("id" %in% names(dat))) dat$id <- NA_character_
    dat <- dat %>% add_id_limpio()

    n_total <- nrow(dat)
    n_id_ok <- sum(!is.na(dat$id_limpio))

    idx <- dat %>%
      filter(!is.na(id_limpio)) %>%
      distinct(id_limpio) %>%
      mutate(file = basename(f), .before = 1)

    fullflat_index[[length(fullflat_index) + 1]] <- idx

    out_parquet <- file.path(
      OUT_FULLFLAT_DIR,
      paste0(tools::file_path_sans_ext(basename(f)), ".parquet")
    )
    arrow::write_parquet(dat, out_parquet)

    fullflat_manifest[[length(fullflat_manifest) + 1]] <- tibble(
      file_csv = f,
      file_parquet = out_parquet,
      n_total = n_total,
      n_con_id = n_id_ok,
      n_cols = ncol(dat)
    )

    rm(dat, idx)
    invisible(gc())
  }

  ff_index_df <- bind_rows(fullflat_index)
  write_csv(ff_index_df, OUT_FULLFLAT_INDEX)

  ff_manif_df <- bind_rows(fullflat_manifest)
  write_csv(ff_manif_df, OUT_FULLFLAT_MANIF)

  cat("\n✅ FULLFLAT convertido a Parquet en:", OUT_FULLFLAT_DIR, "\n")
  cat("🔗 Índice FULLFLAT ids:", OUT_FULLFLAT_INDEX, "(", nrow(ff_index_df), "ids)\n")
  cat("🧾 Manifest FULLFLAT:", OUT_FULLFLAT_MANIF, "\n")
}

# -------------------------
# 5) Guardar log parsing issues (si aplica)
# -------------------------
if (GUARDAR_LOG_PARSING && length(parsing_issues_log) > 0) {
  parsing_df <- bind_rows(parsing_issues_log)

  if (all(c("file","row","col") %in% names(parsing_df))) {
    parsing_df <- parsing_df %>% arrange(file, row, col)
  }

  write_csv(parsing_df, OUT_LOG_PARSING)
  cat("\n🧾 Log parsing issues:", OUT_LOG_PARSING, "\n")
} else if (GUARDAR_LOG_PARSING) {
  cat("\n✅ No parsing issues detectados (problems() vacío).\n")
}

cat("\n📌 Listo: curado unificado en CSV; fullflat quedó como dataset Parquet sin reventar RAM.\n")
cat("   Nota: si hay parsing issues, revisa el log y filtra por columnas repetidas (abstract/affiliations).\n")


📄 Curados encontrados:     10 
📄 FULLFLAT encontrados:    11 
✅ Curado unificado: /content/OA_COVID19_MX_curado_unificado.csv ( 18844 )
🧾 Curado duplicados: /content/OA_COVID19_MX_curado_duplicados.csv ( 0 )

🧱 FULLFLAT -> Parquet: 2019_ncov_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: corona_virus_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: coronavirus_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: covid_19_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: covid19_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: mers_cov_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: middle_east_respiratory_syndrome_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: sars_cov_2_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: sars_cov_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: sars_cov2_OpenAlex_MX_FULLFLAT.csv 

🧱 FULLFLAT -> Parquet: severe_acute_respiratory_syndrome_OpenAlex_MX_FULLFLAT.csv 

✅ FULLFLAT convertido a Parquet en: /content/fullflat_parquet 
🔗 Índice FULLFLAT id

# Explicación detallada del script de transformación OpenAlex → Bibliometrix (Excel + RData)

Este script en **R** toma un dataset de OpenAlex ya “aplanado” (un CSV tipo `df`, con columnas mayormente escalares o listas colapsadas) y lo convierte en dos salidas orientadas a análisis bibliométrico con **Bibliometrix**:

1. **Un archivo Excel** que respeta la estructura/orden de un **template** (plantilla) de campos bibliográficos.
2. **Un archivo `.RData`** que contiene un objeto `M` con clase **`bibliometrixDB`**, listo para cargar en Bibliometrix.

En términos funcionales, el programa hace:  
Lectura CSV → mapeo de campos OpenAlex → normalización (autores, DOI, afiliaciones, listas) → alineación con template → exportación a Excel → exportación a RData → verificación rápida de completitud.

---

## 1) Paquetes y dependencias

El script declara e instala (si faltan) los paquetes:

- **readr**: lectura del CSV de entrada (todo como texto para robustez).
- **dplyr**: transformaciones puntuales y `case_when`.
- **stringr**: limpieza, extracción, sustituciones, normalización de strings.
- **openxlsx**: lectura del template y escritura del Excel final.
- **tibble**: construcción del dataframe final (`out`) de manera consistente.

La instalación se hace solo para paquetes faltantes y luego se cargan en memoria.

---

## 2) Rutas y archivos de entrada/salida

### 2.1 Entrada principal
- `in_csv`: CSV unificado/curado de OpenAlex (ej. `/content/OA_COVID19_MX_curado_unificado.csv`)

### 2.2 Plantilla de Excel
- `template_xlsx`: archivo Excel con la estructura objetivo (columnas Bibliometrix)

El template define el **orden y conjunto** de columnas que el Excel final debe respetar.

### 2.3 Salidas
- `out_xlsx`: Excel final (ej. `/content/COVID19_bibliometrix.xlsx`)
- `out_rdata`: objeto `M` exportado en `.RData` (ej. `/content/COVID19_OpenAlex_M.RData`)

---

## 3) Helpers base: extracción y limpieza

Esta sección crea funciones para volver **determinística** la extracción de columnas y la normalización de valores.

### 3.1 `pick(df, candidates)`
Busca la primera columna existente dentro de `candidates`.  
Si ninguna existe, devuelve `NA` para todas las filas.

Esto hace el script tolerante a variaciones en nombres de columnas (por ejemplo: `ids.doi` vs `ids_doi`).

### 3.2 Limpieza de valores “vacíos disfrazados”
`clean_blankish_vec()` normaliza strings y convierte a `NA` valores como:
- `""`, `"NA"`, `"n/a"`, `"null"`, `"none"`

Esto evita contaminar el output con pseudo-valores.

### 3.3 Coalescencia vectorial
`coalesce_vec(...)` implementa un “coalesce por filas”:  
toma la primera fuente no vacía (según `clean_blankish_vec`) entre múltiples columnas candidatas.

### 3.4 Control de longitud y tipos
- `clip(x, n=30000)`: recorta strings grandes (especialmente abstracts) para evitar límites de Excel o fallos de escritura.
- `to_int(x)`: extrae dígitos y convierte a entero (para `PY`, `TC`, `NR`).
- `clean_doi_vec(x)`: normaliza DOI quitando prefijos comunes (URL de doi.org, `doi:`).

---

## 4) Normalización de autores (AU/AF) y forma “Last, First”

Bibliometrix suele trabajar bien con autores en formato:

`Apellido, Nombre; Apellido, Nombre; ...`

El script implementa:

- Conversión de una cadena de autores (con `;`, `|`, “and/y”, o comas) a un vector consistente.
- Normalización para construir:
  - **AF**: nombres tal como aparecen (concatenados con `; `)
  - **AU**: “Apellido, Nombre” (concatenados con `; `)

### 4.1 Selección robusta de fuente de autores
- Usa `authors` si parece separable.
- Si `authors` luce “no separable” y existe `raw_author_names`, prefiere el raw.

Esto reduce casos donde `authors` viene “aplastado” y no se puede tokenizar correctamente.

### 4.2 Identificadores de autores
`AU_ID` se intenta poblar con columnas tipo `author_ids` o equivalentes.

---

## 5) Lectura del CSV y lectura del template

### 5.1 Lectura del CSV OpenAlex
Se lee con `readr::read_csv` forzando **todas las columnas como `character`**.

Objetivo:
- Evitar inferencias de tipos.
- Prevenir pérdida de ceros a la izquierda.
- Mantener consistencia para exportación a Excel y RData.

### 5.2 Lectura del template
Se lee la hoja 1 del template con `openxlsx::read.xlsx` para recuperar el vector `template_cols`.

Ese vector define el “contrato” de salida: **qué columnas deben existir** y **en qué orden**.

---

## 6) Patch de campos obligatorios (WC, SN, RP, CR)

Antes de construir el output, el script asegura que el template contenga ciertos campos clave:

- **WC**: categorías/áreas temáticas (aquí, derivadas de la taxonomía de Topics de OpenAlex).
- **SN**: identificadores de revista (ISSN).
- **RP**: información de “reprint/corresponding author” (aproximación operacional).
- **CR**: referencias citadas (aquí, basadas en `referenced_works`).

Si el template no los incluye, se agregan al final para no perderlos al reordenar.

---

## 7) Núcleo del mapeo: OpenAlex → tags tipo Bibliometrix

El corazón del script es construir variables con nombres estándar Bibliometrix:

### 7.1 Identificadores y metadatos básicos
- `id_oa`: identificador OpenAlex (coalesce entre posibles columnas)
- `DI`: DOI limpio
- `TI`: título (de `title` o `display_name`)
- `PY`: año de publicación
- `DT`: tipo de documento

### 7.2 Idioma (LA) con mapeo básico ISO → nombre
Convierte códigos comunes (`en`, `es`, `pt`) a nombres (English, Spanish, Portuguese); otros se dejan tal cual.

### 7.3 Abstract (AB)
- Se limpia (espacios, saltos, etc.)
- Se recorta a 30000 caracteres (seguridad para Excel y herramientas posteriores)

### 7.4 Métricas
- `TC`: total de citas (cited_by_count)
- `NR`: número de referencias (referenced_works_count)

### 7.5 Fuente (SO): priorización de revista “real”
`SO` se construye intentando priorizar nombres de fuente más “editoriales” que “repositorios”.

Para ello:
- Se coalescen varias fuentes (`raw_source_name`, `primary_location_source_name`, `locations_raw_source_names`, etc.)
- Se evalúa si la fuente es “mala” (repositorios/archivos/catálogos, plataformas como arXiv/SSRN/Zenodo, etc.)
- Si es “mala”, se intenta reemplazar con una alternativa más plausible.

Esto no es una garantía bibliográfica perfecta, pero mejora la calidad promedio del campo `SO`.

### 7.6 ISSN (SN)
`SN` se forma colapsando múltiples fuentes de ISSN/ISSN-L y limpiando listas semicolon-separadas.

### 7.7 Afiliaciones (C1) e IDs institucionales (C1_ID)
`C1` se arma con:
- `raw_affiliation_strings` o `institutions`

Luego se aplica:
- eliminación de emails y ruido (p.ej. “electronic address”)
- normalización y deduplicación de la lista

`C1_ID` intenta poblarse con IDs institucionales (`institution_ids`, `affiliations_institution_ids`).

### 7.8 Países (C3) e instituciones (AU_UN)
- `C3`: country codes a partir de instituciones o `countries`
- `AU_UN`: instituciones (como lista colapsada)

### 7.9 Keywords y categorías temáticas (DE, ID, WC)
- `DE`: lista de términos (keywords/mesh/concepts/topics) limpia y deduplicada
- `ID`: lista paralela (mesh/concepts/topics)
- `WC`: se define explícitamente como **taxonomía de Topics de OpenAlex** (field/domain/subfield), no categorías WoS oficiales

Incluye mensajes de debug para confirmar que `WC` no está vacío antes de exportar.

### 7.10 URL
Se construye una URL de acceso priorizando:
- landing pages OA
- PDFs cuando existen

---

## 8) RP y CR: aproximación operacional

### 8.1 RP (corresponding author + afiliaciones)
`RP` se construye combinando:

- nombres crudos (`raw_author_names`)
- afiliaciones crudas (`raw_affiliation_strings`)
- máscara `authors_is_corresponding`

Se busca detectar autores marcados como “corresponding”; si no hay, se registra indirectamente si existe `corresponding_author_ids`.

El resultado es un campo textual “operativo” que puede servir como proxy, aunque no sea un RP estándar perfecto estilo WoS/Scopus.

### 8.2 CR (referencias citadas)
`CR` se construye desde `referenced_works` y se normaliza como lista separada por `;`.

Nota: aquí `CR` no es una referencia bibliográfica formateada (autor-año-revista), sino un listado de identificadores/links de works referenciados, lo cual es útil para redes de citación, pero no equivale a CR de WoS “formateado”.

---

## 9) Construcción del dataframe final y alineación con template

### 9.1 Construcción de `out`
Se crea un `tibble` con los campos principales (DI, TI, AU, AF, SO, SN, etc.), más:
- `DB = "OpenAlex"`
- `SR` y `SR_FULL` como `NA` (placeholders comunes en algunos flujos)

### 9.2 Completar columnas faltantes
Se calcula:
- columnas del template que no están en `out`
- se agregan al final como `NA_character_`

### 9.3 Reorden exacto
`out` se reordena con:

`out <- out[, template_cols]`

Así, el Excel queda con el orden esperado por tu plantilla y por flujos downstream.

### 9.4 Limpieza final
Se fuerza todo a `character` y se asegura consistencia de nombres.

---

## 10) Salvaguardas adicionales antes de exportar

### 10.1 `id_oa` nunca vacío
Si falta `id_oa`, se asigna un identificador sintético:

`OA_MISS_1`, `OA_MISS_2`, ...

Esto evita filas sin clave operativa.

### 10.2 Recorte final del abstract
Se recorta `AB` nuevamente por seguridad.

---

## 11) Salida 1: Excel compatible con template

Se crea un workbook nuevo con:

- hoja: `bibliometrix`
- congelamiento de primera fila
- filtros automáticos en encabezados
- escritura completa del dataframe `out2`

Se guarda en `out_xlsx` con overwrite.

Objetivo:
- archivo legible
- columnas consistentes
- listo para inspección manual y/o importación

---

## 12) Salida 2: RData para Bibliometrix (objeto M)

Antes de exportar:
- `PY`, `TC`, `NR` se convierten a enteros (cuando existen)

Luego:
- `M <- out2`
- se asigna clase: `c("bibliometrixDB","data.frame")`
- atributos típicos:
  - `DB = "OpenAlex"`
  - `sep = ";"`

Finalmente se guarda con `save(list="M", file=out_rdata)`.

Esto produce un `.RData` que Bibliometrix puede tratar como base bibliográfica importada.

---

## 13) Mini-check de completitud (sanity checks)

Al final se imprimen contadores de no-vacíos para campos clave:

- AU (y cuántos tienen `;`)
- TI
- SO
- SN
- RP
- CR
- C1
- DE
- WC
- ID

Esto sirve como control rápido para validar que:

- autores están presentes y correctamente tokenizados
- fuente (SO) no está vacía masivamente
- taxonomía temática (WC) está llenando
- listas de keywords y afiliaciones aparecen pobladas

---

## Resultados esperados

Al finalizar la ejecución, `/content/` contendrá:

1. **Excel**: `COVID19_bibliometrix.xlsx`  
   Hoja `bibliometrix` con columnas alineadas al template.

2. **RData**: `COVID19_OpenAlex_M.RData`  
   Objeto `M` clase `bibliometrixDB`, listo para análisis en Bibliometrix.

---

## Garantías metodológicas del script

Garantiza:

- Mapeo consistente y tolerante a nombres variables de columnas.
- Normalización de DOI y autores para formato Bibliometrix.
- Control de strings problemáticos (ruido, emails, largos excesivos).
- Alineación estricta al template (columnas y orden).
- Producción simultánea de salida “humana” (Excel) y salida “analítica” (RData).

No garantiza:

- Equivalencia perfecta con tags WoS/Scopus (especialmente `CR` y `RP`).
- Que `WC` represente categorías oficiales WoS: aquí es explícitamente la taxonomía de Topics de OpenAlex.
- Que `SO` sea siempre una revista “correcta”; se aplica heurística anti-repositorio, no verificación editorial.

In [ ]:
# =========================================================
# OpenAlex FULL tibble/CSV (estilo "df" ya aplanado)
# -> (1) Excel template
# -> (2) RData bibliometrixDB (M)
# =========================================================

pkgs <- c("readr","dplyr","stringr","openxlsx","tibble")
to_install <- pkgs[!pkgs %in% installed.packages()[, "Package"]]
if (length(to_install)) install.packages(to_install, repos="https://cloud.r-project.org")
invisible(lapply(pkgs, library, character.only = TRUE))

# -----------------------------
# RUTAS (ajusta si aplica)
# -----------------------------
in_csv        <- "/content/OA_COVID19_MX_curado_unificado.csv"
template_xlsx <- "/content/openalex_data_20260220_230710.xlsx"
out_xlsx      <- "/content/COVID19_bibliometrix.xlsx"
out_rdata     <- "/content/COVID19_OpenAlex_M.RData"

# -----------------------------
# Helpers base
# -----------------------------
pick <- function(df, candidates) {
  hit <- candidates[candidates %in% names(df)]
  if (!length(hit)) return(rep(NA_character_, nrow(df)))
  as.character(df[[hit[1]]])
}

clean_blankish_vec <- function(x) {
  x <- stringr::str_squish(as.character(x))
  bad <- is.na(x) | x == "" | tolower(x) %in% c("na","n/a","null","none")
  x[bad] <- NA_character_
  x
}

coalesce_vec <- function(...) {
  xs <- list(...)
  xs <- lapply(xs, clean_blankish_vec)
  out <- xs[[1]]
  if (length(xs) > 1) {
    for (k in 2:length(xs)) out <- ifelse(is.na(out), xs[[k]], out)
  }
  out
}

clip <- function(x, n = 30000) {
  x <- as.character(x)
  x <- ifelse(is.na(x), NA_character_, x)
  ifelse(!is.na(x) & nchar(x) > n, substr(x, 1, n), x)
}

to_int <- function(x) {
  x <- as.character(x)
  x <- stringr::str_extract(x, "\\d+")
  suppressWarnings(as.integer(x))
}

clean_doi_vec <- function(x) {
  x <- clean_blankish_vec(x)
  x <- stringr::str_replace_all(x, "^https?://(dx\\.)?doi\\.org/", "")
  x <- stringr::str_replace_all(x, "^doi:\\s*", "")
  x <- stringr::str_squish(x)
  x[x == ""] <- NA_character_
  x
}

# autores: "Nombre Apellido; ..." -> "Apellido, Nombre; ..."
to_last_first_one <- function(name) {
  name <- stringr::str_squish(as.character(name)[1])
  if (is.na(name) || name=="") return(NA_character_)
  parts <- unlist(stringr::str_split(name, "\\s+"))
  if (length(parts) == 1) return(parts)
  last  <- parts[length(parts)]
  first <- paste(parts[-length(parts)], collapse=" ")
  paste0(last, ", ", first)
}

normalize_authors <- function(x_semicolon) {
  x_semicolon <- clean_blankish_vec(x_semicolon)
  vapply(seq_along(x_semicolon), function(i){
    s <- x_semicolon[i]
    if (is.na(s)) return(NA_character_)
    v <- unlist(stringr::str_split(s, "\\s*;\\s*"))
    v <- stringr::str_squish(v); v <- v[nzchar(v)]
    if (!length(v)) return(NA_character_)
    v2 <- vapply(v, to_last_first_one, FUN.VALUE=character(1))
    v2 <- v2[!is.na(v2) & nzchar(v2)]
    if (!length(v2)) return(NA_character_)
    paste(v2, collapse="; ")
  }, FUN.VALUE=character(1))
}

# -----------------------------
# Helpers extra (MEJORAS)
# -----------------------------
clean_cell <- function(x) {
  x <- as.character(x)
  x <- stringr::str_replace_all(x, "[\r\n\t]+", " ")
  x <- stringr::str_squish(x)
  x[x == ""] <- NA_character_
  x
}

clean_semilist <- function(x){
  x <- clean_blankish_vec(x)
  x <- ifelse(is.na(x), NA_character_, x)

  vapply(seq_along(x), function(i){
    s <- x[i]
    if (is.na(s)) return(NA_character_)
    parts <- unlist(stringr::str_split(s, "\\s*;\\s*"))
    parts <- stringr::str_squish(parts)
    parts <- parts[nzchar(parts)]
    if (!length(parts)) return(NA_character_)
    parts <- unique(parts)
    paste(parts, collapse="; ")
  }, FUN.VALUE = character(1))
}

drop_emails_and_noise <- function(x){
  x <- clean_blankish_vec(x)
  x <- ifelse(is.na(x), NA_character_, x)

  x <- stringr::str_replace_all(x, "\\b[[:alnum:]._%+-]+@[[:alnum:].-]+\\.[[:alpha:]]{2,}\\b", " ")
  x <- stringr::str_replace_all(x, "(?i)electronic address\\s*:\\s*", " ")
  x <- stringr::str_replace_all(x, "(?i)e-mail\\s*:\\s*", " ")
  x <- stringr::str_squish(x)
  x[x==""] <- NA_character_
  x
}

first_before_semicolon <- function(x){
  x <- clean_blankish_vec(x)
  vapply(seq_along(x), function(i){
    s <- x[i]
    if (is.na(s)) return(NA_character_)
    parts <- unlist(stringr::str_split(s, "\\s*;\\s*"))
    parts <- stringr::str_squish(parts)
    parts <- parts[nzchar(parts)]
    if (!length(parts)) return(NA_character_)
    parts[1]
  }, FUN.VALUE = character(1))
}

is_bad_source <- function(s){
  s <- tolower(stringr::str_squish(as.character(s)))
  is.na(s) | s=="" |
    s %in% c("pubmed","pubmed central") |
    grepl("repository|institutional|catalog|archive|iris|hal|zenodo|arxiv|ssrn|redalyc|scielo", s)
}

# =========================
# Helpers para RP/CR (NUEVO)
# =========================
split_semicol <- function(x){
  x <- clean_blankish_vec(x)
  lapply(x, function(s){
    if (is.na(s)) return(character(0))
    v <- unlist(stringr::str_split(s, "\\s*;\\s*"))
    v <- stringr::str_squish(v); v <- v[nzchar(v)]
    v
  })
}

# -----------------------------
# Leer datos
# -----------------------------
df <- readr::read_csv(
  in_csv,
  show_col_types = FALSE,
  col_types = readr::cols(.default = readr::col_character())
)

# -----------------------------
# Template columnas
# -----------------------------
template_df   <- openxlsx::read.xlsx(template_xlsx, sheet = 1, colNames = TRUE)
template_cols <- names(template_df)

# =========================================================
# PATCH: asegura que el template incluya campos clave (incluye WC)
# (si no están, los agregamos al final para NO perderlos al reordenar)
# =========================================================
must_have <- c("WC","SN","RP","CR")
missing_must <- setdiff(must_have, template_cols)
if (length(missing_must)) template_cols <- c(template_cols, missing_must)

# -----------------------------
# Núcleo OpenAlex -> tags bibliometrix
# -----------------------------

# IDs / DOI / Título / Año / Tipo
id_oa <- coalesce_vec(
  pick(df, c("id","ids_openalex","ids.openalex","openalex_id"))
)

DI <- clean_doi_vec(coalesce_vec(
  pick(df, c("doi","ids_doi","ids.doi"))
))

TI <- coalesce_vec(
  pick(df, c("title","display_name"))
)

PY <- coalesce_vec(
  pick(df, c("publication_year","year"))
)

DT <- coalesce_vec(
  pick(df, c("type","type_crossref","publication_type"))
)

# -----------------------------
# LA (MEJORADO): ISO -> nombre
# -----------------------------
LA0 <- clean_blankish_vec(coalesce_vec(
  pick(df, c("language","language_code"))
))
LA <- dplyr::case_when(
  LA0 == "en" ~ "English",
  LA0 == "es" ~ "Spanish",
  LA0 == "pt" ~ "Portuguese",
  TRUE ~ LA0
)
LA <- clean_blankish_vec(LA)

# -----------------------------
# AU / AF / AU_ID (bloque robusto)
# -----------------------------
AF_src <- coalesce_vec(
  pick(df, c("raw_author_names")),
  pick(df, c("authors")),
  pick(df, c("author_display_names"))
)

split_authors <- function(s) {
  s <- clean_cell(s)
  if (is.na(s)) return(character(0))

  if (stringr::str_detect(s, ";")) {
    v <- unlist(stringr::str_split(s, "\\s*;\\s*"))
    v <- stringr::str_squish(v); v <- v[nzchar(v)]
    return(v)
  }
  if (stringr::str_detect(s, "\\|")) {
    v <- unlist(stringr::str_split(s, "\\s*\\|\\s*"))
    v <- stringr::str_squish(v); v <- v[nzchar(v)]
    return(v)
  }
  if (stringr::str_detect(stringr::str_to_lower(s), "\\s(and|y)\\s")) {
    v <- unlist(stringr::str_split(s, "\\s+(and|y)\\s+"))
    v <- stringr::str_squish(v); v <- v[nzchar(v)]
    return(v)
  }

  n_commas <- stringr::str_count(s, ",")
  if (n_commas >= 2 && stringr::str_detect(s, "\\w+,\\s*\\w+")) return(s)
  if (n_commas >= 1) {
    v <- unlist(stringr::str_split(s, "\\s*,\\s*"))
    v <- stringr::str_squish(v); v <- v[nzchar(v)]
    return(v)
  }
  return(s)
}

to_last_first <- function(name) {
  name <- clean_cell(name)
  if (is.na(name)) return(NA_character_)
  name <- stringr::str_replace(name, "\\s*-\\s*$", "")
  if (stringr::str_detect(name, ".+?,.+")) return(name)

  parts <- unlist(stringr::str_split(name, "\\s+"))
  parts <- parts[nzchar(parts)]
  if (length(parts) <= 1) return(name)

  last  <- parts[length(parts)]
  first <- paste(parts[-length(parts)], collapse = " ")
  paste0(last, ", ", first)
}

normalize_AF_AU <- function(x) {
  x <- clean_cell(x)
  out_AF <- character(length(x))
  out_AU <- character(length(x))

  for (i in seq_along(x)) {
    v <- split_authors(x[i])
    v <- vapply(v, clean_cell, FUN.VALUE = character(1))
    v <- v[!is.na(v) & nzchar(v)]
    if (!length(v)) {
      out_AF[i] <- NA_character_
      out_AU[i] <- NA_character_
      next
    }
    out_AF[i] <- paste(v, collapse = "; ")
    out_AU[i] <- paste(vapply(v, to_last_first, FUN.VALUE = character(1)), collapse = "; ")
  }
  list(AF = out_AF, AU = out_AU)
}

authors1 <- clean_cell(pick(df, c("authors")))
raw1     <- clean_cell(pick(df, c("raw_author_names")))
looks_unsplittable <- function(s){
  s <- clean_cell(s)
  is.na(s) | (!grepl("[;|,]", s) & !grepl("\\s(and|y)\\s", tolower(s)))
}
use_raw <- looks_unsplittable(authors1) & !is.na(raw1) & nzchar(raw1)
AF_src <- ifelse(use_raw, raw1, AF_src)

norm <- normalize_AF_AU(AF_src)
AF <- norm$AF
AU <- norm$AU

AU_ID <- coalesce_vec(
  pick(df, c("author_ids","authors.id"))
)

# -----------------------------
# AB (limpio + clip)
# -----------------------------
AB <- coalesce_vec(
  pick(df, c("abstract","abstract_raw"))
)
AB <- clean_cell(AB)
AB <- clip(AB, 30000)

# -----------------------------
# TC / NR
# -----------------------------
TC <- coalesce_vec(pick(df, c("cited_by_count")))
NR <- coalesce_vec(pick(df, c("referenced_works_count")))

# -----------------------------
# SO (prioriza revista "real")
# -----------------------------
SO_raw <- coalesce_vec(
  pick(df, c("raw_source_name")),
  pick(df, c("primary_location_source_name")),
  first_before_semicolon(pick(df, c("locations_raw_source_names"))),
  pick(df, c("best_oa_location_raw_source_name")),
  pick(df, c("best_oa_location_source_name")),
  pick(df, c("host_venue")),
  pick(df, c("locations_source_names"))
)
SO <- SO_raw
bad <- vapply(SO, is_bad_source, logical(1))
if (any(bad)) {
  SO2 <- coalesce_vec(
    pick(df, c("raw_source_name")),
    first_before_semicolon(pick(df, c("locations_raw_source_names"))),
    pick(df, c("primary_location_source_name")),
    pick(df, c("best_oa_location_source_name"))
  )
  SO[bad] <- SO2[bad]
}
SO <- clean_blankish_vec(SO)

# -----------------------------
# SN (ISSN)
# -----------------------------
SN <- clean_semilist(coalesce_vec(
  pick(df, c("issn")),
  pick(df, c("issn_l","issn_l_clean","issn_l_code")),
  pick(df, c("host_venue_issn","host_venue_issn_l")),
  pick(df, c("primary_location_source_issn","primary_location_source_issn_l")),
  pick(df, c("best_oa_location_source_issn","best_oa_location_source_issn_l")),
  pick(df, c("locations_source_issn","locations_source_issn_l"))
))

# -----------------------------
# C1 (limpia afiliaciones + dedup lista)
# -----------------------------
C1 <- coalesce_vec(
  pick(df, c("raw_affiliation_strings")),
  pick(df, c("institutions"))
)
C1 <- drop_emails_and_noise(C1)
C1 <- clean_semilist(C1)

C1_ID <- coalesce_vec(
  pick(df, c("institution_ids")),
  pick(df, c("affiliations_institution_ids"))
)

# Countries (C3) e instituciones (AU_UN)
C3 <- coalesce_vec(
  pick(df, c("institution_country_codes")),
  pick(df, c("countries"))
)

AU_UN <- coalesce_vec(
  pick(df, c("institutions"))
)

# -----------------------------
# DE / ID (listas limpias ; dedup)
# -----------------------------
DE <- clean_semilist(coalesce_vec(
  pick(df, c("keywords")),
  pick(df, c("mesh_terms")),
  pick(df, c("concepts")),
  pick(df, c("topics"))
))

ID <- clean_semilist(coalesce_vec(
  pick(df, c("mesh_terms")),
  pick(df, c("concepts")),
  pick(df, c("topics"))
))

# -----------------------------
# WC (MEJORADO): usa OpenAlex Topics (NO WoS Categories)
# Nota analítica: WC aquí representa "OpenAlex Topic taxonomy" (domain/field/subfield),
# no categorías oficiales de Web of Science.
# -----------------------------
WC <- clean_semilist(coalesce_vec(
  pick(df, c("topics_field_names")),
  pick(df, c("topics_domain_names")),
  pick(df, c("topics_subfield_names")),
  pick(df, c("primary_topic_field_name")),
  pick(df, c("primary_topic_domain_name")),
  pick(df, c("primary_topic_subfield_name")),
  pick(df, c("topics"))
))

# DEBUG RÁPIDO: confirma si WC se está calculando antes del template
message("WC sample: ", paste(head(WC[!is.na(WC)], 3), collapse=" || "))
message("WC non-empty pre-out: ", sum(!is.na(WC) & trimws(WC)!=""))

# URL
URL <- coalesce_vec(
  pick(df, c("best_oa_location_url","primary_location_landing_page_url","open_access_url","primary_location_landing_page_url")),
  pick(df, c("primary_location_landing_page_url")),
  pick(df, c("best_oa_location_pdf_url","primary_location_pdf_url"))
)

# =========================================================
# RP (operacional) + CR (básico)
# =========================================================

raw_names <- pick(df, c("raw_author_names"))
raw_affs  <- pick(df, c("raw_affiliation_strings"))
mask_corr <- pick(df, c("authors_is_corresponding"))

names_list <- split_semicol(raw_names)
affs_list  <- split_semicol(raw_affs)
mask_list  <- split_semicol(mask_corr)

RP <- vapply(seq_len(nrow(df)), function(i){
  nms <- names_list[[i]]
  msk <- tolower(mask_list[[i]])
  aff <- affs_list[[i]]

  corr_idx <- which(msk %in% c("true","t","1","yes","y"))
  corr_nms <- if (length(corr_idx) && length(nms)) nms[corr_idx] else character(0)

  if (!length(corr_nms)) {
    ca <- clean_blankish_vec(pick(df, c("corresponding_author_ids")))[i]
    if (!is.na(ca)) corr_nms <- "(corresponding_author_ids present)"
  }

  aff2 <- aff
  if (length(aff2) > 3) aff2 <- aff2[1:3]

  if (!length(corr_nms) && !length(aff2)) return(NA_character_)

  paste0(
    if (length(corr_nms)) paste0("CORRESPONDING: ", paste(corr_nms, collapse="; ")) else "",
    if (length(corr_nms) && length(aff2)) " | " else "",
    if (length(aff2)) paste0("AFF: ", paste(unique(aff2), collapse="; ")) else ""
  )
}, FUN.VALUE = character(1))

RP <- clean_blankish_vec(RP)

CR <- clean_semilist(pick(df, c("referenced_works")))

# -----------------------------
# Construir OUT
# -----------------------------
out <- tibble::tibble(
  id_oa = id_oa,
  DI = DI,
  TI = TI,
  LA = LA,
  PY = PY,
  DT = DT,
  SO = SO,
  SN = SN,
  AU = AU,
  AF = AF,
  AU_ID = AU_ID,
  AB = AB,
  DE = DE,
  ID = ID,
  WC = WC,
  TC = TC,
  NR = NR,
  C1 = C1,
  C1_ID = C1_ID,
  C3 = C3,
  AU_UN = AU_UN,
  RP = RP,
  CR = CR,
  URL = URL,
  DB = "OpenAlex",
  SR = NA_character_,
  SR_FULL = NA_character_
)

# Completa con columnas del template y orden exacto
missing_in_out <- setdiff(template_cols, names(out))
if (length(missing_in_out) > 0) for (cc in missing_in_out) out[[cc]] <- NA_character_

# Mantén el orden del template (ya con WC/SN/RP/CR asegurados)
out <- out[, template_cols]

# Limpieza general
out2 <- as.data.frame(out, stringsAsFactors = FALSE)
rownames(out2) <- NULL
out2[] <- lapply(out2, function(col){ col <- as.character(col); names(col) <- NULL; col })

# id_oa nunca vacío
out2$id_oa <- ifelse(is.na(out2$id_oa) | out2$id_oa == "",
                     paste0("OA_MISS_", seq_len(nrow(out2))),
                     out2$id_oa)

# recortes por seguridad
if ("AB" %in% names(out2)) out2$AB <- clip(out2$AB, 30000)

# -----------------------------
# (1) Excel
# -----------------------------
wb <- openxlsx::createWorkbook()
openxlsx::addWorksheet(wb, "bibliometrix")
openxlsx::writeData(wb, "bibliometrix", out2, rowNames = FALSE)
openxlsx::freezePane(wb, "bibliometrix", firstRow = TRUE)
openxlsx::addFilter(wb, "bibliometrix", row = 1, cols = 1:ncol(out2))
openxlsx::saveWorkbook(wb, out_xlsx, overwrite = TRUE)

# -----------------------------
# (2) RData: bibliometrixDB (M)
# -----------------------------
if ("PY" %in% names(out2)) out2$PY <- to_int(out2$PY)
if ("TC" %in% names(out2)) out2$TC <- to_int(out2$TC)
if ("NR" %in% names(out2)) out2$NR <- to_int(out2$NR)

M <- out2
rownames(M) <- as.character(seq_len(nrow(M)))
class(M) <- c("bibliometrixDB","data.frame")
attr(M, "DB")  <- "OpenAlex"
attr(M, "sep") <- ";"

save(list="M", file=out_rdata)

message("OK Excel -> ", out_xlsx)
message("OK RData -> ", out_rdata)

# -----------------------------
# Mini-check: completitud clave
# -----------------------------
nonempty <- function(x) sum(!is.na(x) & trimws(as.character(x))!="")
message("AU non-empty: ", nonempty(M$AU))
message("AU with ';' : ", sum(!is.na(M$AU) & grepl(";", M$AU, fixed=TRUE)))
message("TI non-empty: ", nonempty(M$TI))
message("SO non-empty: ", nonempty(M$SO))
message("SN non-empty: ", if ("SN" %in% names(M)) nonempty(M$SN) else 0)
message("RP non-empty: ", if ("RP" %in% names(M)) nonempty(M$RP) else 0)
message("CR non-empty: ", if ("CR" %in% names(M)) nonempty(M$CR) else 0)
message("C1 non-empty: ", nonempty(M$C1))
message("DE non-empty: ", nonempty(M$DE))
message("WC non-empty: ", if ("WC" %in% names(M)) nonempty(M$WC) else 0)
message("ID non-empty: ", if ("ID" %in% names(M)) nonempty(M$ID) else 0)


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

WC sample: Mathematics; Medicine || Medicine; Biochemistry, Genetics and Molecular Biology; Chemistry || Agricultural and Biological Sciences; Pharmacology, Toxicology and Pharmaceutics

WC non-empty pre-out: 18578

OK Excel -> /content/COVID19_bibliometrix.xlsx

OK RData -> /content/COVID19_OpenAlex_M.RData

AU non-empty: 18844

AU with ';' : 16205

TI non-empty: 18817

SO non-empty: 17750

SN non-empty: 15557

RP non-empty: 18844

CR non-empty: 14677

C1 non-empty: 18842

DE non-empty: 18696

WC non-empty: 18578

ID non-empty: 18696



# Explicación detallada del script OpenAlex: FULLFLAT “Nano-like” + modelo relacional + KV lossless (streaming anti-RAM, optimizado + PATCH HTTR)

Este script en **R** implementa un pipeline de extracción y transformación de OpenAlex orientado a **alto volumen** (decenas de miles de works) con tres productos simultáneos y complementarios, pero con una ingeniería más agresiva para reducir tiempo y riesgos de RAM:

1. **FULLFLAT “Nano-like”**: una tabla ancha (1 fila por work) con bloques “wide” (autores, instituciones, conceptos, referencias, etc.) con **topes** para evitar explosión de columnas.  
2. **Modelo relacional (Opción B)**: un conjunto de tablas normalizadas (works, authorships, instituciones, conceptos, referencias, etc.) conectadas por `id_url`.  
3. **KV lossless (key–value)**: una tabla que serializa **toda** la estructura del JSON en forma `(id_url, path, index, value_json)` para conservar información sin pérdida, incluso si no se modeló explícitamente.

La escritura es **incremental (streaming)**, pero en esta versión se optimiza el I/O:  
en vez de escribir *por work*, se acumula *por chunk* y se escribe **una vez por tabla por chunk**, usando `data.table::fwrite`.

No se genera XLSX por diseño: Excel es el “depredador natural” de la RAM (y del optimismo).

---

## 0) Dependencias y configuración inicial

### 0.1 Paquetes
Se instalan/cargan:

- **openalexR**: se mantiene en dependencias por compatibilidad, pero la descarga principal se parchea con `httr`.
- **httr**: **nuevo**. Se usa para consultar OpenAlex directo y evitar errores HTTP específicos del wrapper en ciertos entornos (ver bloque 2).
- **dplyr / purrr / stringr / tibble**: manipulación, mapeo funcional y bitácora.
- **readr / jsonlite**: serialización JSON (clave para KV lossless).
- **data.table**: escritura rápida a CSV (`fwrite`) y consolidaciones eficientes (`rbindlist`).
- **openxlsx**: se conserva, aunque el script evita XLSX.

### 0.2 Paralelización de escritura
`data.table::setDTthreads(0)` permite que `data.table` use el máximo de hilos disponibles (si el runtime lo permite).

### 0.3 Configuración de cortesía del API
`options(openalexR.mailto = "tu_correo@dominio.com")` se usa como `mailto` en los requests (buenas prácticas del API / polite pool).

### 0.4 Operador `%||%`
Operador de fallback para nulos/vacíos: evita cascadas de `if (is.null(...))`.

---

## 1) Parámetros del pipeline

### 1.1 Términos de búsqueda
`terminos` contiene variantes de COVID-19/coronavirus para buscar en `title_and_abstract.search`.

### 1.2 Directorio de salida
`out_dir <- "/content/OA_relacional_out"`  
Ahí se generan los CSV de cada tabla.

### 1.3 Topes del FULLFLAT
Para controlar el ancho del FULLFLAT:

- `MAX_AUTHORS` (100)
- `MAX_REFS` (1354)
- `MAX_CONCEPTS` (38)
- `MAX_RELATED` (20)
- `MAX_CBY` (14)
- `MAX_GRANTS` (50)

Esto produce un esquema “hoja gigante” pero **predecible**.

### 1.4 Paginación y chunking
- `per_page <- 200`: tamaño de página del API.
- `chunk_size <- 100`: bloque de works procesados por iteración.

`chunk_size` es tu “perilla” principal:
- más alto: menos overhead, más velocidad, más RAM;
- más bajo: más estable, más lento.

### 1.5 KV_FLUSH_ROWS
Nuevo parámetro operativo:
- `KV_FLUSH_ROWS <- 200000`

El KV lossless se **bufferiza** y se “flushea” a disco cuando supera ese umbral, para no reventar RAM (el KV suele ser el monstruo final boss).

---

## 2) Descarga robusta (LIST crudo) con **PATCH HTTR**

### 2.1 Por qué se cambia el método de descarga
En ciertos entornos (ej. RStudio con configuraciones particulares, proxys, etc.), `openalexR::oa_fetch()` puede fallar con:

- `HTTP 400 Request Line is too large`

Esto suele estar asociado al modo en que se construye/transporta la URL (o headers) por el wrapper.

**Solución:** se implementa una descarga directa con `httr::GET()` usando la paginación por cursor nativa del API de OpenAlex.

### 2.2 Función `oa_fetch_list_retry()` (PATCH)
Ahora esta función:

- construye `base_url <- "https://api.openalex.org/works"`
- arma el filtro:  
  `filter = "title_and_abstract.search:<term>,authorships.countries:MX"`
- pagina con:
  - `cursor="*"` inicial
  - `meta$next_cursor` para avanzar
- agrega `mailto` si está disponible
- valida status HTTP y parsea JSON con `httr::content(..., as="parsed")`

### 2.3 Reintentos (retry) y guardrails
- Hasta `tries = 6`.
- Backoff exponencial (`2^k`, cap 60s).
- Guardrail de páginas (`page_guard`) para evitar loops infinitos por cursor.

### 2.4 Acumulación por término + bitácora
Por cada término:
- descarga lista de works (cada work es una lista)
- agrega `search_term` al work
- acumula en `all_works`
- registra en `log_tbl` el `n_results`

### 2.5 Deduplicación global por `id`
Se deduplica globalmente:
- extrae `id` de cada work
- elimina duplicados entre términos
- salida: `works_dedup`

Resultado típico:
- `works totales` (sumatoria por término)
- `works únicos` (deduplicados)

---

## 3) Helpers de normalización y serialización

### 3.1 Abstract
`decode_abstract_inverted()` reconstruye el abstract desde `abstract_inverted_index` cuando `abstract` no viene como texto.

### 3.2 Colapsos a string
- `collapse_chr()`: lista/vector → string con separador (`;`)
- `affil_one()`: vuelve escalar la afiliación cruda (evita columnas listas)

### 3.3 JSON barato (pero válido)
Se reemplaza `json_str()` por `json_fast()`:

- si es `NULL` → `"null"`
- si es atómico escalar → `toJSON(auto_unbox=TRUE)`
- si es lista/objeto → `toJSON(...)`
- fallback a `NA_character_` si hay error

Esto reduce costo sin sacrificar el hecho de que el valor siga siendo JSON válido.

---

## 4) KV lossless ITERATIVO (más rápido que recursivo)

### 4.1 Objetivo
El KV lossless es el “seguro metodológico”:
conserva el JSON completo aunque no se modele explícitamente en relacional o FULLFLAT.

### 4.2 Estructura de salida
Cada fila representa un nodo del JSON:

- `id_url`: work id
- `path`: ruta (ej. `host_venue.display_name`)
- `index`: posición para listas (o NA si escalar)
- `value_json`: valor serializado

### 4.3 Cambio clave: `kv_walk_iter()`
Antes: recorrido recursivo → overhead alto.  
Ahora: recorrido **iterativo con stack**:

- Evita profundidad recursiva.
- Acelera el recorrido.
- Normaliza tipos antes de consolidar con `rbindlist`.
- Usa `ignore.attr=TRUE` para evitar choques de atributos/clases.

El resultado puede ser enorme; por eso se combina con el buffer + flush (sección 7).

---

## 5) Modelo relacional por work (Opción B)

### 5.1 `work_to_relational(w)`
Construye un paquete de tablas conectadas por `id_url`:

- `works`: metadatos core, OA, venue, ids, urls, abstract, search_term
- `authorships`: 1 fila por autoría
- `authorship_institutions`: 1 fila por institución por autoría
- `concepts`: 1 fila por concepto
- `references`: 1 fila por referenced_work
- `related_works`: 1 fila por related_work
- `counts_by_year`: 1 fila por año
- `grants`: 1 fila por grant
- `topics`: 1 fila por topic
- `kv`: la tabla lossless del work (por `kv_walk_iter`)

### 5.2 Resultado
Devuelve lista de data.tables lista para ser agregada al chunk.

---

## 6) FULLFLAT “Nano-like” (1 fila por work)

### 6.1 Qué contiene
Construye una fila ancha con:

- Campos “tipo bibliometrix” (AU, TI, AB, SO, DT, PY, TC, DI, CR, etc.)
- Campos OpenAlex core (ids, OA, venue, urls)
- Campos agregados derivados:
  - `AU`, `C1`, `AU_UN`, `AU_CO`, `ID`

### 6.2 WIDE con topes
Se crean columnas parametrizadas hasta el máximo:

- Autores (100) + 1 institución principal por autor
- Referencias (1354)
- Related (20)
- Conceptos (38)
- counts_by_year (14 pares)
- grants (50)

Esto es **FULLFLAT** por diseño: útil para análisis plano, pero controlado por topes.

---

## 7) Escritura incremental POR CHUNK (optimización mayor)

### 7.1 Cambio sustancial
Antes: `fwrite()` por cada work y por tabla → muchísimo overhead (I/O constante).  
Ahora:

- Para cada chunk:
  - se juntan filas en listas (`CH_works`, `CH_auth`, etc.)
  - al final del chunk se hace `rbindlist()` y **un solo `fwrite()` por tabla**

Esto reduce drásticamente tiempo total.

### 7.2 Función `write_dt()`
Wrapper para:
- no escribir si `dt` está vacío
- escribir con `append`
- poner headers solo la primera vez (`col.names = !file.exists(file)`)

### 7.3 Buffer + flush del KV (crítico)
El KV lossless no se acumula infinito:

- `kv_buffer` acumula filas de KV
- cuando llega a `KV_FLUSH_ROWS`:
  - se escribe al CSV
  - se vacía buffer
  - `gc()`

Esto mantiene RAM estable aunque el KV sea gigantesco.

### 7.4 Truncado de abstract (seguridad práctica)
Se recorta a 32000 caracteres en:
- `works$abstract`
- `FULLFLAT$AB`

No es “por Excel” (porque aquí no se hace XLSX), sino para evitar textos monstruosos que rompan herramientas downstream.

---

## 8) Empaquetado final en ZIP (TODOS los CSV)

### 8.1 Objetivo
Convierte el directorio completo de salida en un artefacto único:

- descarga más fácil en Colab
- menos riesgo de “me faltó un CSV”
- snapshot consistente de la corrida

### 8.2 Implementación
- `zip_path <- file.path(out_dir, "OA_relacional_out_CSVs.zip")`
- `csvs <- list.files(out_dir, pattern="\\.csv$", full.names=TRUE)`
- si ya existe el ZIP → se reemplaza
- se hace `setwd(out_dir)` para meter rutas relativas (ZIP portable)
- `zip(zipfile=basename(zip_path), files=basename(csvs))`

### 8.3 Auditoría
Mensajes finales:
- ruta del ZIP
- número de CSV incluidos

---

## 9) Salidas esperadas (estructura final)

En `out_dir` se generan:

### 9.1 Relacional
- `works_covid.csv` *(nota: en tu script se renombró así)*
- `authorships.csv`
- `authorship_institutions.csv`
- `concepts.csv`
- `references.csv`
- `related_works.csv`
- `counts_by_year.csv`
- `grants.csv`
- `topics.csv`

### 9.2 KV lossless
- `COVID19_kv_lossless.csv`

### 9.3 FULLFLAT Nano-like
- `COVID19_FULLFLAT_Nano_like.csv`

### 9.4 ZIP final
- `OA_relacional_out_CSVs.zip`

Además se imprime el `log_tbl` con conteo de resultados por término.

---

## 10) Garantías y limitaciones

### 10.1 Garantías
Este pipeline garantiza:

- Descarga robusta por cursor con reintentos.
- Deduplicación global por `id`.
- Triple salida simultánea:
  - FULLFLAT controlado por topes
  - relacional normalizado para redes/joins
  - KV lossless para preservación total
- Escritura optimizada (1 write por tabla por chunk).
- KV estable vía buffer + flush.
- ZIP final para portabilidad.

### 10.2 Limitaciones
- **KV lossless** puede producir archivos masivos (muchos millones de filas).
- FULLFLAT trunca excedentes por topes (por diseño).
- CSV gigantes no son para Excel: Excel se ahoga rápido.
- El tiempo total depende más de I/O y tamaño de KV que de CPU.

---

## Resultado conceptual

Este script produce un “triángulo” de datos:

1. **FULLFLAT Nano-like**: análisis plano rápido + “hoja gigante” controlada.  
2. **Relacional**: redes (coautoría/instituciones/citación), joins limpios, escalable.  
3. **KV lossless**: auditoría y recuperación completa del JSON, sin reconsultar el API.

Y al final, el ZIP te deja todo en un solo paquete, como si tu pipeline dijera:  
“no soy Excel, pero sí soy ordenado”.

In [ ]:
# =========================================================
# FULLFLAT (Nano-like) + RELACIONAL (Opción B) + KV lossless
# OPTIMIZADO:
# - escritura por CHUNKS (1 write por tabla por chunk)
# - KV lossless con serialización más barata (sin perder info)
# - flush de KV por volumen para no reventar RAM
# - usa threads de data.table en fwrite
# - ZIP final
# =========================================================

# -----------------------------
# 0) Paquetes
# -----------------------------
pkgs <- c("openalexR","dplyr","purrr","stringr","tibble","openxlsx","readr","jsonlite","data.table","httr")
to_install <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(to_install)) install.packages(to_install, repos="https://cloud.r-project.org")
invisible(lapply(pkgs, library, character.only = TRUE))

# Usa más hilos (si Colab da)
data.table::setDTthreads(0)

options(openalexR.mailto = "tu_correo@dominio.com")

`%||%` <- function(a, b) if (!is.null(a) && length(a) > 0 && !all(is.na(a))) a else b

# -----------------------------
# 1) Parámetros
# -----------------------------
terminos <- c(
  "COVID-19", "COVID19", "Coronavirus", "Corona virus", "2019-nCoV",
  "SARS-CoV2", "SARS-CoV-2", "SARS-CoV", "MERS-CoV",
  "Severe Acute Respiratory Syndrome", "Middle East Respiratory Syndrome"
)

out_dir <- "/content/OA_relacional_out"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

MAX_AUTHORS   <- 100
MAX_REFS      <- 1354
MAX_CONCEPTS  <- 38
MAX_RELATED   <- 20
MAX_CBY       <- 14
MAX_GRANTS    <- 50

per_page   <- 200
chunk_size <- 100      # si se traba, baja a 10; si vuela, sube 50

# KV es monstruoso: flush cuando pase este umbral de filas
KV_FLUSH_ROWS <- 200000

# -----------------------------
# 2) Descarga robusta (LIST crudo)  **PATCH HTTR**
# - Evita "HTTP 400 Request Line is too large" de openalexR en algunos entornos
# - Cursor pagination nativa de OpenAlex
# - Devuelve: lista de works (cada work es una lista)
# -----------------------------
oa_fetch_list_retry <- function(search_term, per_page = 200, tries = 6, verbose = FALSE) {

  search_term <- stringr::str_squish(search_term)

  base_url <- "https://api.openalex.org/works"
  mailto   <- getOption("openalexR.mailto", NA_character_)

  # filtro OpenAlex (ojo con comas)
  filter_str <- paste0("title_and_abstract.search:", search_term, ",authorships.countries:MX")

  for (k in seq_len(tries)) {

    cursor <- "*"
    out_results <- list()
    page_guard <- 0L

    ok <- tryCatch({

      repeat {
        page_guard <- page_guard + 1L
        if (page_guard > 200000L) stop("Guardrail: demasiadas páginas (cursor loop).")

        q <- list(
          filter   = filter_str,
          `per-page` = per_page,
          cursor   = cursor
        )
        if (!is.na(mailto) && nzchar(mailto)) q$mailto <- mailto

        resp <- httr::GET(
          url = base_url,
          query = q,
          httr::user_agent("OA-pipeline/1.0 (httr)"),
          httr::timeout(180)
        )

        sc <- httr::status_code(resp)
        if (sc >= 400) {
          txt <- tryCatch(httr::content(resp, as = "text", encoding = "UTF-8"), error = function(e) "")
          stop(sprintf("HTTP status %s %s", sc, trimws(txt)))
        }

        dat <- httr::content(resp, as = "parsed", type = "application/json", encoding = "UTF-8")

        # resultados
        if (!is.null(dat$results) && length(dat$results) > 0) {
          out_results <- c(out_results, dat$results)
        }

        # siguiente cursor
        next_cursor <- dat$meta$next_cursor %||% NULL
        if (is.null(next_cursor) || !nzchar(as.character(next_cursor))) break

        cursor <- as.character(next_cursor)
      }

      TRUE
    }, error = function(e) e)

    if (!inherits(ok, "error")) {
      if (length(out_results) > 0 && is.list(out_results[[1]]) && !is.null(out_results[[1]]$id)) {
        return(out_results)
      } else {
        # sin results reales
        ok <- "sin results"
      }
    }

    wait <- min(60, 2^k)
    msg <- if (inherits(ok, "error")) conditionMessage(ok) else as.character(ok)
    message(sprintf("⚠️ intento %d/%d falló (%s). Reintento en %ds", k, tries, msg, wait))
    Sys.sleep(wait)
  }

  return(NULL)
}

cat("📥 Descargando works (LIST crudo) por término...\n")
all_works <- list()
log_tbl <- tibble::tibble(termino=character(), n_results=integer())

for (t in terminos) {
  cat("\n🔍", t, "\n")
  w <- oa_fetch_list_retry(t, per_page = per_page, tries = 6, verbose = FALSE)
  n <- if (is.null(w)) 0L else length(w)
  log_tbl <- dplyr::bind_rows(log_tbl, tibble::tibble(termino=t, n_results=n))
  if (n == 0L) next
  w <- purrr::map(w, ~{ .x$search_term <- t; .x })
  all_works <- c(all_works, w)
  Sys.sleep(1)
}

if (length(all_works) == 0) stop("No se descargó nada. Revisa red/mailto/filtros.")

ids <- purrr::map_chr(all_works, ~ as.character(.x$id %||% NA_character_))
keep <- !is.na(ids) & !duplicated(ids)
works_dedup <- all_works[keep]
cat("\n✅ Works totales:", length(all_works), " | únicos:", length(works_dedup), "\n")

# -----------------------------
# 3) Helpers: abstract + collapse + JSON-safe
# -----------------------------
decode_abstract_inverted <- function(inv_index) {
  if (is.null(inv_index)) return(NA_character_)
  inv_values <- unlist(inv_index, use.names = FALSE)
  if (length(inv_values) == 0) return(NA_character_)
  words <- character(max(inv_values) + 1)
  for (word in names(inv_index)) for (i in inv_index[[word]]) words[i + 1] <- word
  paste(words, collapse = " ")
}

collapse_chr <- function(x, sep=";") {
  if (is.null(x) || length(x) == 0) return(NA_character_)
  x <- unlist(x, recursive = TRUE, use.names = FALSE)
  x <- x[!is.na(x)]
  if (!length(x)) return(NA_character_)
  paste(as.character(x), collapse = sep)
}

affil_one <- function(a, sep = ";") {
  x <- a$raw_affiliation_string
  if (is.null(x) || length(x) == 0 || all(is.na(x))) x <- a$raw_affiliation_strings
  collapse_chr(x, sep = sep)
}

# JSON rápido: evita toJSON cuando no hace falta (pero sigue siendo JSON válido)
json_fast <- function(x) {
  if (is.null(x)) return("null")
  if (is.atomic(x) && length(x) == 1) {
    return(jsonlite::toJSON(x, auto_unbox = TRUE, null = "null"))
  }
  out <- tryCatch(jsonlite::toJSON(x, auto_unbox = TRUE, null = "null"), error = function(e) NA_character_)
  as.character(out)
}

# -----------------------------
# 4) KV lossless ITERATIVO (más rápido que recursivo)
# (work_id, path, index, value_json)
# -----------------------------
kv_walk_iter <- function(x) {Limitaciones
  stack <- list(list(obj = x, path = "", idx = NA_integer_))
  out <- vector("list", 0L)

  push <- function(obj, path, idx) {
    if (!is.na(idx)) idx <- as.integer(idx)
    stack[[length(stack) + 1L]] <<- list(obj = obj, path = path, idx = idx)
  }

  while (length(stack)) {
    cur <- stack[[length(stack)]]
    stack[[length(stack)]] <- NULL

    obj  <- cur$obj
    path <- as.character(cur$path)
    idx  <- cur$idx
    if (!is.na(idx)) idx <- as.integer(idx)

    if (is.null(obj)) {
      out[[length(out) + 1L]] <- list(
        path = path,
        index = idx,
        value_json = "null"
      )
      next
    }

    if (!is.list(obj) || is.data.frame(obj)) {
      if (is.atomic(obj) && length(obj) > 1) {
        for (i in seq_along(obj)) {
          out[[length(out) + 1L]] <- list(
            path = path,
            index = as.integer(i),
            value_json = json_fast(obj[[i]])
          )
        }
      } else {
        out[[length(out) + 1L]] <- list(
          path = path,
          index = idx,
          value_json = json_fast(obj)
        )
      }
      next
    }

    nms <- names(obj)
    if (!is.null(nms) && any(nzchar(nms))) {
      for (nm in rev(nms)) {
        child <- obj[[nm]]
        child_path <- if (path == "") nm else paste0(path, ".", nm)
        push(child, child_path, NA_integer_)
      }
    } else {
      for (i in rev(seq_along(obj))) {
        child <- obj[[i]]
        child_path <- paste0(path, "[", i, "]")
        push(child, child_path, as.integer(i))
      }
    }
  }

  for (k in seq_along(out)) {
    out[[k]]$path <- as.character(out[[k]]$path)
    out[[k]]$index <- ifelse(is.na(out[[k]]$index), NA_integer_, as.integer(out[[k]]$index))
    out[[k]]$value_json <- as.character(out[[k]]$value_json)
  }

  data.table::rbindlist(out, fill = TRUE, ignore.attr = TRUE)
}

# -----------------------------
# 5) Tablas relacionales por work (Opción B)
# -----------------------------
work_to_relational <- function(w) {
  work_id <- w$id %||% NA_character_

  abs_txt <- w$abstract %||% NA_character_
  if (is.null(abs_txt) || length(abs_txt) == 0 || all(is.na(abs_txt))) {
    abs_txt <- decode_abstract_inverted(w$abstract_inverted_index %||% NULL)
  }
  abs_txt <- as.character(abs_txt)

  host <- w$host_venue %||% list()
  oa   <- w$open_access %||% list()
  ids  <- w$ids %||% list()
  pri  <- w$primary_location %||% list()

  works_dt <- data.table::data.table(
    id_url = work_id,
    title = w$title %||% NA_character_,
    doi = w$doi %||% NA_character_,
    publication_year = w$publication_year %||% NA_integer_,
    publication_date = w$publication_date %||% NA_character_,
    type = w$type %||% NA_character_,
    cited_by_count = w$cited_by_count %||% NA_integer_,
    cited_by_api_url = w$cited_by_api_url %||% NA_character_,
    language = w$language %||% NA_character_,
    is_paratext = w$is_paratext %||% NA,
    is_retracted = w$is_retracted %||% NA,
    abstract = abs_txt,
    search_term = w$search_term %||% NA_character_,

    ids_openalex = ids$openalex %||% NA_character_,
    ids_doi      = ids$doi %||% NA_character_,
    ids_mag      = ids$mag %||% NA_character_,
    ids_pmid     = ids$pmid %||% NA_character_,
    ids_pmcid    = ids$pmcid %||% NA_character_,

    is_oa = oa$is_oa %||% NA,
    is_oa_anywhere = oa$is_oa_anywhere %||% NA,
    oa_status = oa$oa_status %||% NA_character_,
    oa_url = oa$oa_url %||% NA_character_,
    any_repository_has_fulltext = oa$any_repository_has_fulltext %||% NA,

    so_id = host$id %||% NA_character_,
    SO = host$display_name %||% NA_character_,
    issn_l = host$issn_l %||% NA_character_,
    host_organization = host$host_organization %||% NA_character_,

    url = pri$landing_page_url %||% NA_character_,
    pdf_url = pri$pdf_url %||% NA_character_,
    license = pri$license %||% NA_character_,
    version = pri$version %||% NA_character_,
    first_page = pri$first_page %||% NA_character_,
    last_page  = pri$last_page %||% NA_character_,
    volume = pri$volume %||% NA_character_,
    issue  = pri$issue %||% NA_character_
  )

  auth <- w$authorships %||% list()
  authorships_dt <- data.table::rbindlist(lapply(seq_along(auth), function(i) {
    a <- auth[[i]]
    data.table::data.table(
      id_url = work_id,
      author_order = i,
      author_id = a$author$id %||% NA_character_,
      author_display_name = a$author$display_name %||% NA_character_,
      author_orcid = a$author$orcid %||% NA_character_,
      author_position = a$author_position %||% NA_character_,
      is_corresponding = a$is_corresponding %||% NA,
      raw_affiliation = affil_one(a, sep=";")
    )
  }), fill = TRUE)

  auth_inst_dt <- data.table::rbindlist(lapply(seq_along(auth), function(i) {
    a <- auth[[i]]
    insts <- a$institutions %||% list()
    if (!length(insts)) return(NULL)
    data.table::rbindlist(lapply(seq_along(insts), function(j) {
      inst <- insts[[j]]
      data.table::data.table(
        id_url = work_id,
        author_order = i,
        inst_order = j,
        institution_id = inst$id %||% NA_character_,
        institution_display_name = inst$display_name %||% NA_character_,
        institution_ror = inst$ror %||% NA_character_,
        institution_country_code = inst$country_code %||% NA_character_,
        institution_type = inst$type %||% NA_character_,
        institution_lineage = collapse_chr(inst$lineage, sep=";")
      )
    }), fill = TRUE)
  }), fill = TRUE)

  con <- w$concepts %||% list()
  concepts_dt <- data.table::rbindlist(lapply(seq_along(con), function(i) {
    c <- con[[i]]
    data.table::data.table(
      id_url = work_id,
      concept_order = i,
      concept_id = c$id %||% NA_character_,
      concept_wikidata = c$wikidata %||% NA_character_,
      concept_display_name = c$display_name %||% NA_character_,
      concept_level = c$level %||% NA_integer_,
      concept_score = c$score %||% NA_real_
    )
  }), fill = TRUE)

  refs <- w$referenced_works %||% list()
  refs_dt <- data.table::rbindlist(lapply(seq_along(refs), function(i) {
    data.table::data.table(id_url = work_id, ref_order = i, referenced_work_id = as.character(refs[[i]]))
  }), fill = TRUE)

  rel <- w$related_works %||% list()
  rel_dt <- data.table::rbindlist(lapply(seq_along(rel), function(i) {
    data.table::data.table(id_url = work_id, related_order = i, related_work_id = as.character(rel[[i]]))
  }), fill = TRUE)

  cby <- w$counts_by_year %||% list()
  cby_dt <- data.table::rbindlist(lapply(seq_along(cby), function(i) {
    yy <- cby[[i]]
    data.table::data.table(
      id_url = work_id,
      cby_order = i,
      year = yy$year %||% NA_integer_,
      cited_by_count = yy$cited_by_count %||% NA_integer_
    )
  }), fill = TRUE)

  gr <- w$grants %||% list()
  grants_dt <- data.table::rbindlist(lapply(seq_along(gr), function(i) {
    g <- gr[[i]]
    data.table::data.table(
      id_url = work_id,
      grant_order = i,
      award_id = g$award_id %||% NA_character_,
      funder = g$funder %||% NA_character_,
      funder_display_name = g$funder_display_name %||% NA_character_
    )
  }), fill = TRUE)

  top <- w$topics %||% list()
  topics_dt <- data.table::rbindlist(lapply(seq_along(top), function(i) {
    tt <- top[[i]]
    data.table::data.table(
      id_url = work_id,
      topic_order = i,
      topic_id = tt$id %||% NA_character_,
      topic_display_name = tt$display_name %||% NA_character_,
      topic_score = tt$score %||% NA_real_,
      subfield_id = tt$subfield$id %||% NA_character_,
      subfield_display_name = tt$subfield$display_name %||% NA_character_,
      field_id = tt$field$id %||% NA_character_,
      field_display_name = tt$field$display_name %||% NA_character_,
      domain_id = tt$domain$id %||% NA_character_,
      domain_display_name = tt$domain$display_name %||% NA_character_
    )
  }), fill = TRUE)

  # KV lossless
  kv_dt <- kv_walk_iter(w)
  kv_dt[, id_url := work_id]
  data.table::setcolorder(kv_dt, c("id_url","path","index","value_json"))

  list(
    works = works_dt,
    authorships = authorships_dt,
    auth_inst = auth_inst_dt,
    concepts = concepts_dt,
    references = refs_dt,
    related = rel_dt,
    counts_by_year = cby_dt,
    grants = grants_dt,
    topics = topics_dt,
    kv = kv_dt
  )
}

# -----------------------------
# 6) FULLFLAT Prueba04-like (1 fila por work)
# -----------------------------
work_to_fullflat_prueba04 <- function(w) {
  abs_txt <- w$abstract %||% NA_character_
  if (is.null(abs_txt) || length(abs_txt) == 0 || all(is.na(abs_txt))) {
    abs_txt <- decode_abstract_inverted(w$abstract_inverted_index %||% NULL)
  }
  abs_txt <- as.character(abs_txt)

  host <- w$host_venue %||% list()
  oa   <- w$open_access %||% list()
  ids  <- w$ids %||% list()
  pri  <- w$primary_location %||% list()
  auth <- w$authorships %||% list()

  au_vec <- vapply(auth, function(a) a$author$display_name %||% NA_character_, character(1))
  AU <- paste(au_vec[!is.na(au_vec)], collapse = ";")
  if (!nzchar(AU)) AU <- NA_character_

  cor_vec <- vapply(auth, function(a) as.character(a$is_corresponding %||% NA), character(1))
  author_is_corresponding <- paste(cor_vec, collapse = ", ")

  raw_aff <- vapply(auth, function(a) affil_one(a, sep=";"), character(1))
  C1 <- paste(unique(raw_aff[!is.na(raw_aff) & nzchar(raw_aff)]), collapse = ";")
  if (!nzchar(C1)) C1 <- NA_character_

  insts_all <- unlist(lapply(auth, function(a) lapply(a$institutions %||% list(), `[[`, "display_name")),
                      recursive = TRUE, use.names = FALSE)
  AU_UN <- paste(unique(insts_all[!is.na(insts_all) & nzchar(insts_all)]), collapse = ";")
  if (!nzchar(AU_UN)) AU_UN <- NA_character_

  inst_co <- unlist(lapply(auth, function(a) lapply(a$institutions %||% list(), `[[`, "country_code")),
                    recursive = TRUE, use.names = FALSE)
  AU_CO <- paste(unique(inst_co[!is.na(inst_co) & nzchar(inst_co)]), collapse = ";")
  if (!nzchar(AU_CO)) AU_CO <- NA_character_

  con <- w$concepts %||% list()
  con_names <- vapply(con, function(c) c$display_name %||% NA_character_, character(1))
  ID <- paste(unique(con_names[!is.na(con_names)]), collapse = ";")
  if (!nzchar(ID)) ID <- NA_character_

  refs <- w$referenced_works %||% list()
  CR <- if (length(refs)) paste(as.character(refs), collapse = ";") else NA_character_

  topics_str <- if (!is.null(w$topics)) collapse_chr(w$topics, sep=", ") else NA_character_

  row <- list(
    AU = toupper(AU %||% NA_character_),
    RP = NA_character_,
    C1 = toupper(C1 %||% NA_character_),
    AU_UN = toupper(AU_UN %||% NA_character_),
    AU_CO = AU_CO %||% NA_character_,
    ID = toupper(ID %||% NA_character_),

    id_url = w$id %||% NA_character_,
    title = w$title %||% NA_character_,

    author_is_corresponding = author_is_corresponding %||% NA_character_,
    publication_date = w$publication_date %||% NA_character_,

    so_id = host$id %||% NA_character_,
    host_organization = host$host_organization %||% NA_character_,
    issn_l = host$issn_l %||% NA_character_,

    url = pri$landing_page_url %||% NA_character_,
    pdf_url = pri$pdf_url %||% NA_character_,
    license = pri$license %||% NA_character_,
    version = pri$version %||% NA_character_,
    first_page = pri$first_page %||% NA_character_,
    last_page  = pri$last_page %||% NA_character_,
    volume = pri$volume %||% NA_character_,
    issue  = pri$issue %||% NA_character_,

    is_oa = oa$is_oa %||% NA,
    is_oa_anywhere = oa$is_oa_anywhere %||% NA,
    oa_status = oa$oa_status %||% NA_character_,
    oa_url = oa$oa_url %||% NA_character_,
    any_repository_has_fulltext = oa$any_repository_has_fulltext %||% NA,

    language = w$language %||% NA_character_,
    cited_by_api_url = w$cited_by_api_url %||% NA_character_,

    ids_openalex = ids$openalex %||% NA_character_,
    ids_doi      = ids$doi %||% NA_character_,
    ids_mag      = ids$mag %||% NA_character_,
    ids_pmid     = ids$pmid %||% NA_character_,
    ids_pmcid    = ids$pmcid %||% NA_character_,

    doi = w$doi %||% NA_character_,
    is_paratext = w$is_paratext %||% NA,
    is_retracted = w$is_retracted %||% NA,
    topics = topics_str %||% NA_character_,

    TI = toupper(w$title %||% NA_character_),
    AB = toupper(abs_txt %||% NA_character_),
    SO = toupper(host$display_name %||% NA_character_),
    DT = toupper(w$type %||% NA_character_),
    DB = "OPENALEX",
    PY = w$publication_year %||% NA_integer_,
    TC = w$cited_by_count %||% NA_integer_,
    DI = toupper((ids$doi %||% w$doi) %||% NA_character_),
    CR = toupper(CR %||% NA_character_)
  )

  # (Se mantiene tu bloque WIDE exactamente igual)
  nauth <- min(length(auth), MAX_AUTHORS)
  for (i in seq_len(MAX_AUTHORS)) {
    row[[paste0("author_au_id_", i)]]              <- NA_character_
    row[[paste0("author_au_display_name_", i)]]    <- NA_character_
    row[[paste0("author_au_orcid_", i)]]           <- NA_character_
    row[[paste0("author_author_position_", i)]]    <- NA_character_
    row[[paste0("author_au_affiliation_raw_", i)]] <- NA_character_
    row[[paste0("author_institution_id_", i)]]           <- NA_character_
    row[[paste0("author_institution_display_name_", i)]] <- NA_character_
    row[[paste0("author_institution_ror_", i)]]          <- NA_character_
    row[[paste0("author_institution_country_code_", i)]] <- NA_character_
    row[[paste0("author_institution_type_", i)]]         <- NA_character_
    row[[paste0("author_institution_lineage_", i)]]      <- NA_character_
  }
  if (nauth > 0) {
    for (i in seq_len(nauth)) {
      a <- auth[[i]]
      row[[paste0("author_au_id_", i)]]           <- a$author$id %||% NA_character_
      row[[paste0("author_au_display_name_", i)]] <- a$author$display_name %||% NA_character_
      row[[paste0("author_au_orcid_", i)]]        <- a$author$orcid %||% NA_character_
      row[[paste0("author_author_position_", i)]] <- a$author_position %||% NA_character_
      row[[paste0("author_au_affiliation_raw_", i)]] <- affil_one(a, sep=";")
      insts <- a$institutions %||% list()
      inst1 <- if (length(insts) >= 1) insts[[1]] else NULL
      if (!is.null(inst1)) {
        row[[paste0("author_institution_id_", i)]]           <- inst1$id %||% NA_character_
        row[[paste0("author_institution_display_name_", i)]] <- inst1$display_name %||% NA_character_
        row[[paste0("author_institution_ror_", i)]]          <- inst1$ror %||% NA_character_
        row[[paste0("author_institution_country_code_", i)]] <- inst1$country_code %||% NA_character_
        row[[paste0("author_institution_type_", i)]]         <- inst1$type %||% NA_character_
        row[[paste0("author_institution_lineage_", i)]]      <- collapse_chr(inst1$lineage, sep=";")
      }
    }
  }

  refs2 <- w$referenced_works %||% list()
  nrefs <- min(length(refs2), MAX_REFS)
  for (i in seq_len(MAX_REFS)) row[[paste0("referenced_works_", i)]] <- NA_character_
  if (nrefs > 0) for (i in seq_len(nrefs)) row[[paste0("referenced_works_", i)]] <- as.character(refs2[[i]])

  rel2 <- w$related_works %||% list()
  nrel <- min(length(rel2), MAX_RELATED)
  for (i in seq_len(MAX_RELATED)) row[[paste0("related_works_", i)]] <- NA_character_
  if (nrel > 0) for (i in seq_len(nrel)) row[[paste0("related_works_", i)]] <- as.character(rel2[[i]])

  con2 <- w$concepts %||% list()
  ncon <- min(length(con2), MAX_CONCEPTS)
  for (i in seq_len(MAX_CONCEPTS)) {
    row[[paste0("concepts_id_", i)]]           <- NA_character_
    row[[paste0("concepts_wikidata_", i)]]     <- NA_character_
    row[[paste0("concepts_display_name_", i)]] <- NA_character_
    row[[paste0("concepts_level_", i)]]        <- NA_integer_
    row[[paste0("concepts_score_", i)]]        <- NA_real_
  }
  if (ncon > 0) {
    for (i in seq_len(ncon)) {
      c <- con2[[i]]
      row[[paste0("concepts_id_", i)]]           <- c$id %||% NA_character_
      row[[paste0("concepts_wikidata_", i)]]     <- c$wikidata %||% NA_character_
      row[[paste0("concepts_display_name_", i)]] <- c$display_name %||% NA_character_
      row[[paste0("concepts_level_", i)]]        <- c$level %||% NA_integer_
      row[[paste0("concepts_score_", i)]]        <- c$score %||% NA_real_
    }
  }

  cby2 <- w$counts_by_year %||% list()
  ncby <- min(length(cby2), MAX_CBY)
  for (i in seq_len(MAX_CBY)) {
    row[[paste0("counts_by_year_year_", i)]] <- NA_integer_
    row[[paste0("counts_by_year_cited_by_count_", i)]] <- NA_integer_
  }
  if (ncby > 0) {
    for (i in seq_len(ncby)) {
      yy <- cby2[[i]]
      row[[paste0("counts_by_year_year_", i)]] <- yy$year %||% NA_integer_
      row[[paste0("counts_by_year_cited_by_count_", i)]] <- yy$cited_by_count %||% NA_integer_
    }
  }

  gr <- w$grants %||% list()
  ngr <- min(length(gr), MAX_GRANTS)
  for (i in seq_len(MAX_GRANTS)) {
    row[[paste0("grants_award_id_", i)]] <- NA_character_
    row[[paste0("grants_funder_", i)]] <- NA_character_
    row[[paste0("grants_funder_display_name_", i)]] <- NA_character_
  }
  if (ngr > 0) {
    for (i in seq_len(ngr)) {
      g <- gr[[i]]
      row[[paste0("grants_award_id_", i)]] <- g$award_id %||% NA_character_
      row[[paste0("grants_funder_", i)]] <- g$funder %||% NA_character_
      row[[paste0("grants_funder_display_name_", i)]] <- g$funder_display_name %||% NA_character_
    }
  }

  row
}

# -----------------------------
# 7) Escritura incremental POR CHUNK (mucho más rápido)
# -----------------------------
f_works    <- file.path(out_dir, "works_covid.csv")
f_auth     <- file.path(out_dir, "authorships.csv")
f_ainst    <- file.path(out_dir, "authorship_institutions.csv")
f_con      <- file.path(out_dir, "concepts.csv")
f_refs     <- file.path(out_dir, "references.csv")
f_rel      <- file.path(out_dir, "related_works.csv")
f_cby      <- file.path(out_dir, "counts_by_year.csv")
f_gr       <- file.path(out_dir, "grants.csv")
f_topics   <- file.path(out_dir, "topics.csv")
f_kv       <- file.path(out_dir, "COVID19_kv_lossless.csv")
f_fullflat <- file.path(out_dir, "COVID19_FULLFLAT_Nano_like.csv")

for (ff in c(f_works,f_auth,f_ainst,f_con,f_refs,f_rel,f_cby,f_gr,f_topics,f_kv,f_fullflat)) {
  if (file.exists(ff)) file.remove(ff)
}

write_dt <- function(dt, file) {
  if (is.null(dt) || nrow(dt) == 0) return(invisible(NULL))
  data.table::fwrite(dt, file, append = file.exists(file), col.names = !file.exists(file))
  invisible(NULL)
}

n <- length(works_dedup)
cat("\n🧱 Procesando en chunks (STREAMING optimizado)...\n",
    "   total works:", n, " | chunk_size:", chunk_size, "\n", sep="")

t0 <- Sys.time()

# buffer KV para flush
kv_buffer <- NULL

for (start in seq(1, n, by = chunk_size)) {
  end <- min(n, start + chunk_size - 1)
  cat(sprintf("\n📦 Chunk %d-%d (%d works) | %s\n",
              start, end, end-start+1, format(Sys.time(), "%H:%M:%S")))

  slice <- works_dedup[start:end]

  # coleccionadores por chunk
  CH_works  <- list(); CH_auth <- list(); CH_ainst <- list(); CH_con <- list()
  CH_refs   <- list(); CH_rel  <- list(); CH_cby  <- list(); CH_gr  <- list()
  CH_topics <- list(); CH_ff   <- list()

  for (w in slice) {
    rel <- work_to_relational(w)

    # truncar abstract por límite Excel (seguridad práctica)
    if ("abstract" %in% names(rel$works)) {
      tl <- !is.na(rel$works$abstract) & nchar(rel$works$abstract) > 32000
      if (any(tl)) rel$works$abstract[tl] <- substr(rel$works$abstract[tl], 1, 32000)
    }

    CH_works[[length(CH_works)+1]] <- rel$works
    if (nrow(rel$authorships)) CH_auth[[length(CH_auth)+1]] <- rel$authorships
    if (nrow(rel$auth_inst))   CH_ainst[[length(CH_ainst)+1]] <- rel$auth_inst
    if (nrow(rel$concepts))    CH_con[[length(CH_con)+1]] <- rel$concepts
    if (nrow(rel$references))  CH_refs[[length(CH_refs)+1]] <- rel$references
    if (nrow(rel$related))     CH_rel[[length(CH_rel)+1]] <- rel$related
    if (nrow(rel$counts_by_year)) CH_cby[[length(CH_cby)+1]] <- rel$counts_by_year
    if (nrow(rel$grants))      CH_gr[[length(CH_gr)+1]] <- rel$grants
    if (nrow(rel$topics))      CH_topics[[length(CH_topics)+1]] <- rel$topics

    # KV lossless: buffer + flush por volumen
    if (nrow(rel$kv)) {
      if (is.null(kv_buffer)) kv_buffer <- rel$kv else kv_buffer <- data.table::rbindlist(list(kv_buffer, rel$kv), fill=TRUE)
      if (nrow(kv_buffer) >= KV_FLUSH_ROWS) {
        cat("   💾 Flush KV (", nrow(kv_buffer), " filas)\n", sep="")
        write_dt(kv_buffer, f_kv)
        kv_buffer <- NULL
        invisible(gc())
      }
    }

    # FULLFLAT row
    ff_row <- work_to_fullflat_prueba04(w)
    df_ff <- data.table::as.data.table(ff_row)

    if ("AB" %in% names(df_ff)) {
      tl2 <- !is.na(df_ff$AB) & nchar(df_ff$AB) > 32000
      if (any(tl2)) df_ff$AB[tl2] <- substr(df_ff$AB[tl2], 1, 32000)
    }
    CH_ff[[length(CH_ff)+1]] <- df_ff

    rm(rel, ff_row, df_ff)
    invisible(gc())
  }

  # escribir una vez por tabla por chunk
  write_dt(data.table::rbindlist(CH_works,  fill=TRUE), f_works)
  write_dt(data.table::rbindlist(CH_auth,   fill=TRUE), f_auth)
  write_dt(data.table::rbindlist(CH_ainst,  fill=TRUE), f_ainst)
  write_dt(data.table::rbindlist(CH_con,    fill=TRUE), f_con)
  write_dt(data.table::rbindlist(CH_refs,   fill=TRUE), f_refs)
  write_dt(data.table::rbindlist(CH_rel,    fill=TRUE), f_rel)
  write_dt(data.table::rbindlist(CH_cby,    fill=TRUE), f_cby)
  write_dt(data.table::rbindlist(CH_gr,     fill=TRUE), f_gr)
  write_dt(data.table::rbindlist(CH_topics, fill=TRUE), f_topics)
  write_dt(data.table::rbindlist(CH_ff,     fill=TRUE), f_fullflat)

  rm(slice, CH_works, CH_auth, CH_ainst, CH_con, CH_refs, CH_rel, CH_cby, CH_gr, CH_topics, CH_ff)
  invisible(gc())
}

# flush final de KV si quedó algo
if (!is.null(kv_buffer) && nrow(kv_buffer)) {
  cat("💾 Flush final KV (", nrow(kv_buffer), " filas)\n", sep="")
  write_dt(kv_buffer, f_kv)
  kv_buffer <- NULL
}

cat("\n✅ CSVs relacionales + FULLFLAT + kv_lossless creados en:\n", out_dir, "\n", sep="")
cat("⏱️ Tiempo total:", round(difftime(Sys.time(), t0, units="mins"), 2), "min\n")
cat("🧾 LOG:\n"); print(log_tbl)

# -----------------------------
# 8) Empaquetar TODOS los CSV en un ZIP
# -----------------------------
zip_path <- file.path(out_dir, "OA_relacional_out_CSVs.zip")
csvs <- list.files(out_dir, pattern = "\\.csv$", full.names = TRUE)

if (!length(csvs)) {
  message("⚠️ No hay CSVs en out_dir para comprimir: ", out_dir)
} else {
  if (file.exists(zip_path)) file.remove(zip_path)

  oldwd <- getwd()
  setwd(out_dir)
  on.exit(setwd(oldwd), add = TRUE)

  zip(zipfile = basename(zip_path), files = basename(csvs))
  message("✅ ZIP creado: ", zip_path)
  message("📦 Archivos incluidos: ", length(csvs))
}


📥 Descargando works (LIST crudo) por término...

🔍 COVID-19 

🔍 COVID19 

🔍 Coronavirus 

🔍 Corona virus 

🔍 2019-nCoV 

🔍 SARS-CoV2 

🔍 SARS-CoV-2 

🔍 SARS-CoV 

🔍 MERS-CoV 

🔍 Severe Acute Respiratory Syndrome 

🔍 Middle East Respiratory Syndrome 

✅ Works totales: 30470  | únicos: 19024 

🧱 Procesando en chunks (STREAMING)...
   total works:19024 | chunk_size:25
